This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [1]:
#!pip install keras keras-hub --upgrade -q

In [2]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [3]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## Language models and the Transformer

이 장에서는 다음 내용을 다룹니다.

* 딥러닝 모델을 이용한 **텍스트 생성** 방법
* 영어를 스페인어로 **번역하는 모델** 학습
* 텍스트 모델링 문제를 위한 강력한 아키텍처, **트랜스포머**

이전 장에서 텍스트 전처리 및 모델링의 기초를 다룬 후, 이 장에서는 **기계 번역과 같은 좀 더 복잡한 언어 문제**를 다룹니다. ChatGPT와 같은 제품에 적용되어 자연어 처리(NLP) 분야에 대한 투자를 촉발시킨 트랜스포머 모델에 대한 탄탄한 이해를 쌓아갈 것입니다.

### The language model

이전 장에서는 텍스트 데이터를 숫자 입력으로 변환하는 방법을 배우고, 이 숫자 표현을 사용하여 영화 리뷰를 분류했습니다. 하지만 텍스트 분류는 여러 면에서 매우 간단한 문제입니다. 이진 분류의 경우 하나의 부동 소수점 숫자만 출력하면 되고, N개 변수 분류의 경우에도 최악의 경우 N개의 숫자만 출력하면 됩니다.

그렇다면 **질문 답변이나 번역과 같은 다른 텍스트 기반 작업**은 어떨까요? 많은 실제 문제에서 우리는 주어진 입력에 대한 텍스트 출력을 생성할 수 있는 모델에 관심이 있습니다. 모델에 텍스트를 입력하기 위해 토크나이저와 임베딩이 필요했던 것처럼, 모델에서 텍스트를 출력하기 전에도 몇 가지 기술을 구축해야 합니다.

여기서 처음부터 시작할 필요는 없습니다. **텍스트를 자연스럽게 숫자로 표현하는 정수 시퀀스라는 개념을 계속 사용**할 수 있습니다. 이전 장에서는 입력을 토큰으로 분할하고 각 토큰을 정수로 매핑하는 문자열 토큰화 방법을 다뤘습니다. **시퀀스를 역으로 토큰화하려면 정수를 다시 문자열 토큰으로 매핑하고 이들을 결합**하면 됩니다. 이러한 접근 방식을 사용하면, 우리의 **문제는 토큰의 정수 시퀀스를 예측할 수 있는 모델을 구축**하는 것으로 귀결됩니다.

**가장 간단한 방법은 가능한 모든 출력 정수 시퀀스 공간에 대해 직접 분류기를 학습**시키는 것이지만, 간단한 계산만으로도 이것이 **현실적으로 불가능**하다는 것을 알 수 있습니다. 20,000개의 단어로 이루어진 어휘라면, 20,000^4, 즉 160경 개의 가능한 4단어 시퀀스가 존재하며, 우주의 원자 수보다 20단어로 이루어진 시퀀스의 수가 더 많습니다. 모든 출력 시퀀스를 고유한 분류기 출력으로 표현하려고 시도하는 것은 모델을 어떻게 설계하든 컴퓨팅 자원을 한계까지 밀어붙일 것입니다.

이러한 예측 문제를 실현 가능하게 만드는 **실용적인 접근 방식은 한 번에 하나의 토큰 출력만 예측하는 모델을 구축**하는 것입니다. **언어 모델**은 가장 간단한 형태로, 단순하지만 심오한 **확률 분포인 p(토큰|이전 토큰)을 학습하는 모델**입니다. 특정 시점까지 관찰된 모든 토큰 시퀀스가 주어졌을 때, 언어 모델은 다음에 올 수 있는 모든 가능한 토큰에 대한 확률 분포를 출력하려고 시도합니다. 20,000개의 단어 어휘를 가진 모델은 20,000개의 출력만 예측하면 되지만, 다음 토큰을 반복적으로 예측함으로써 긴 텍스트 시퀀스를 생성할 수 있는 모델을 구축할 수 있습니다.

이를 좀 더 구체적으로 이해하기 위해, **문자열 시퀀스에서 다음 문자를 예측하는 간단한 언어 모델**을 만들어 보겠습니다. 셰익스피어풍의 텍스트를 출력할 수 있는 작은 모델을 학습시켜 보겠습니다.

#### Training a Shakespeare language model

우선, 셰익스피어의 희곡과 소네트 모음집을 다운로드할 수 있습니다.

In [4]:
import keras

filename = keras.utils.get_file(
    origin=(
        "https://storage.googleapis.com/download.tensorflow.org/"
        "data/shakespeare.txt"
    ),
)
shakespeare = open(filename, "r").read()

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


몇 가지 데이터를 살펴보겠습니다.

In [5]:
print(shakespeare[:250])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



이 입력값을 기반으로 언어 모델을 구축하려면 원본 텍스트를 가공해야 합니다. 먼저, 시계열 분석에서 날씨 측정값을 처리했던 것처럼 **데이터를 동일한 길이의 덩어리로 나누어 배치 처리하고 모델 학습에 사용**합니다. 문자 단위 토크나이저를 사용하기 때문에 문자열 입력에 직접 덩어리 분할을 적용할 수 있습니다. **100자 문자열은 100개의 정수로 이루어진 시퀀스**로 변환됩니다.

또한 **각 입력값을 두 개의 개별 특징 시퀀스와 레이블 시퀀스로 분리**합니다. **각 레이블 시퀀스는 입력 시퀀스에서 한 문자만큼 오프셋된 값**입니다.

In [6]:
import tensorflow as tf

sequence_length = 100

def split_input(input, sequence_length):
    for i in range(0, len(input), sequence_length):
        yield input[i : i + sequence_length]

features = list(split_input(shakespeare[:-1], sequence_length))
labels = list(split_input(shakespeare[1:], sequence_length))
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

(x, y) 입력 샘플을 살펴보겠습니다. 시퀀스의 각 위치에 대한 레이블은 시퀀스에서 다음 문자입니다.

In [7]:
x, y = next(dataset.as_numpy_iterator())
x[:50], y[:50]

(b'First Citizen:\nBefore we proceed any further, hear',
 b'irst Citizen:\nBefore we proceed any further, hear ')

이 입력을 정수 시퀀스로 매핑하기 위해 지난 장에서 살펴본 TextVectorization 레이어를 다시 사용할 수 있습니다. 단어 수준 어휘 대신 문자 수준 어휘를 학습하려면 split 인수를 변경할 수 있습니다. 기본값인 "공백" 분할 대신 "문자"를 기준으로 분할합니다. 여기서는 **표준화를 수행하지 않고 간단하게 유지하기 위해 대소문자를 유지하고 구두점은 변경하지 않고 그대로 전달**합니다.

In [8]:
from keras import layers

tokenizer = layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)
tokenizer.adapt(dataset.map(lambda text, labels: text))

어휘를 살펴보겠습니다.

In [9]:
vocabulary_size = tokenizer.vocabulary_size()
vocabulary_size

67

**전체 원문을 처리하는 데 필요한 문자는 단 67개**뿐입니다.

다음으로, 입력 텍스트에 토큰화 레이어를 적용할 수 있습니다. 마지막으로, 데이터셋을 섞고, 배치 처리하고, 캐싱하여 매 에포크마다 다시 계산할 필요가 없도록 할 수 있습니다.

In [10]:
dataset = dataset.map(
    lambda features, labels: (tokenizer(features), tokenizer(labels)),
    num_parallel_calls=8,
)
training_data = dataset.shuffle(10_000).batch(64).cache()

이제 모델링을 시작할 준비가 되었습니다.

간단한 언어 모델을 구축하기 위해, 우리는 **과거의 모든 문자를 기반으로 특정 문자의 확률을 예측**하고자 합니다. 이 책에서 살펴본 다양한 모델링 방법 중에서 **RNN(순환 신경망)이 가장 적합**합니다. 각 셀의 순환 상태를 통해 과거 문자에 대한 정보를 현재 문자의 레이블을 예측하는 데 활용할 수 있기 때문입니다. 또한 이전 장에서 살펴본 것처럼 **임베딩을 사용하여 각 입력 문자를 고유한 256차원 벡터로 임베딩**할 수도 있습니다.

모델의 크기를 줄이고 학습을 용이하게 하기 위해 **순환 레이어는 하나만 사용**하겠습니다. 어떤 순환 레이어든 사용할 수 있지만, 간단하게 **GRU를 사용**하겠습니다. **GRU는 속도가 빠르고 LSTM보다 내부 상태가 간단**합니다.

<p style="text-align:center">
<img src="rnn.png" width="500"><br>Figure 15.0: Example character-level language RNN. The input and label sequences are machin and achine, respectively.</p>

In [11]:
embedding_dim = 256
hidden_dim = 1024

inputs = layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")
x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = layers.GRU(hidden_dim, return_sequences=True)(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)

모델 요약을 살펴보겠습니다.

In [12]:
model.summary(line_length=80)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)            │ (None, 100)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ embedding (Embedding)             │ (None, 100, 256)         │        17,152 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ gru (GRU)                         │ (None, 100, 1024)        │     3,938,304 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout (Dropout)                 │ (None, 100, 1024)        │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense (Dense)                     │ (None, 100, 67)          │        68,675 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 4,024,131 (15.35 MB)

 Trainable params: 4,024,131 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

이 모델은 어휘에 있는 모든 문자에 대해 소프트맥스 확률을 출력하며, 크로스엔트로피 손실 함수를 사용하여 컴파일합니다. 이 모델은 여전히 ​​분류 문제를 학습하는 것이지만, 시퀀스의 각 토큰에 대해 하나의 분류 예측을 수행합니다. 100개의 문자로 구성된 64개의 샘플 배치에 대해 6,400개의 개별 레이블을 예측합니다. 학습 중에 Keras에서 보고되는 손실 및 정확도 지표는 먼저 각 시퀀스별로, 그리고 두 번째로 각 배치별로 평균을 냅니다.

이제 언어 모델 학습을 시작해 보겠습니다.

In [13]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.fit(training_data, epochs=20)

Epoch 1/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 17s 78ms/step - loss: 2.6989 - sparse_categorical_accuracy: 0.2790
Epoch 2/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 66ms/step - loss: 1.9813 - sparse_categorical_accuracy: 0.4197
Epoch 3/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 67ms/step - loss: 1.7139 - sparse_categorical_accuracy: 0.4911
Epoch 4/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 68ms/step - loss: 1.5659 - sparse_categorical_accuracy: 0.5302
Epoch 5/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 21s 69ms/step - loss: 1.4747 - sparse_categorical_accuracy: 0.5536
Epoch 6/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 71ms/step - loss: 1.4104 - sparse_categorical_accuracy: 0.5704
Epoch 7/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 72ms/step - loss: 1.3610 - sparse_categorical_accuracy: 0.5825
Epoch 8/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 75ms/step - loss: 1.3167 - sparse_categorical_accuracy: 0.5935
Epoch 9/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 14s 79ms/step - loss: 1.2759 - sparse_categorical_accuracy: 0.6041
Epoch 10/20
175/175 ━━━━━━━━━━━━━━━━━

20번의 에포크를 거친 후, 우리 모델은 입력 시퀀스에서 다음 문자를 **약 70%의 확률로 예측**할 수 있게 되었습니다.

#### Generating Shakespeare

이제 개별 토큰을 어느 정도 정확하게 예측할 수 있는 모델을 학습시켰으므로, 이를 사용하여 전체 예측 시퀀스를 외삽해 보고자 합니다. 이를 위해 **모델을 반복문에서 호출하여, 한 시점의 모델 예측 출력을 다음 시점의 모델 입력으로 사용**할 수 있습니다. 이러한 피드백 루프를 위해 구축된 모델을 **자기회귀(autoregressive) 모델**이라고 합니다.

이러한 루프를 실행하려면 방금 학습시킨 **모델을 약간 수정해야** 합니다. 학습 과정에서는 모델이 100개의 토큰으로 구성된 고정된 시퀀스 길이만 처리했고, GRU 셀의 상태는 레이어를 호출할 때 암묵적으로 전달되었습니다. 하지만 **생성 과정에서는 한 번에 하나의 출력 토큰만 예측하고 GRU 셀의 상태를 명시적으로 출력해야** 합니다. **모델이 과거 입력 문자에 대해 인코딩한 모든 정보를 담고 있는 이 상태를 다음 호출 시점에 전달**해야 합니다.

이제 **한 번에 하나의 입력 문자만 처리하고 RNN 상태를 명시적으로 전달할 수 있는 모델**을 만들어 보겠습니다. **이 모델은 입력과 출력이 약간 수정된 것을 제외하고는 동일한 계산 구조**를 가지므로, 한 모델의 가중치를 다른 모델에 할당할 수 있습니다.

In [14]:
inputs = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")

x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state = layers.GRU(hidden_dim, return_state=True)(
    x, initial_state=input_state
)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
generation_model = keras.Model(
    inputs=(inputs, input_state),
    outputs=(outputs, output_state),
)
generation_model.set_weights(model.get_weights())

이렇게 하면 루프를 통해 모델을 호출하여 출력 시퀀스를 예측할 수 있습니다. 그 전에, 문자에서 정수로 전환하고 프롬프트(새로운 토큰 예측을 시작하기 전에 모델에 입력할 텍스트 조각)를 선택하기 위해 명시적인 조회 테이블을 만들겠습니다.

In [15]:
tokens = tokenizer.get_vocabulary()
token_ids = range(vocabulary_size)
char_to_id = dict(zip(tokens, token_ids))
id_to_char = dict(zip(token_ids, tokens))

prompt = """
KING RICHARD III:
"""

응답 생성을 시작하려면 먼저 프롬프트를 사용하여 GRU의 내부 상태를 "준비"해야 합니다. 이를 위해 프롬프트를 토큰 단위로 모델에 입력합니다. 이렇게 하면 학습 중에 해당 프롬프트를 만났을 때 모델이 보게 될 정확한 RNN 상태를 계산할 수 있습니다.

프롬프트의 마지막 문자를 모델에 입력하면 상태 출력에 전체 프롬프트 시퀀스에 대한 정보가 포함됩니다. 최종 출력 예측값을 저장해 두면 나중에 생성된 응답의 첫 번째 문자를 선택하는 데 사용할 수 있습니다.

In [16]:
input_ids = [char_to_id[c] for c in prompt]
state = keras.ops.zeros(shape=(1, hidden_dim))
for token_id in input_ids:
    inputs = keras.ops.expand_dims([token_id], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

이제 모델이 새로운 출력 시퀀스를 예측하도록 할 준비가 되었습니다. **원하는 길이까지 반복문을 사용하여 모델이 예측한 다음 문자 중 가장 가능성이 높은 문자를 지속적으로 선택**하고, 이를 **모델에 입력한 다음, 새로운 RNN 상태를 저장**합니다. 이러한 방식으로 전체 시퀀스를 **한 번에 하나의 토큰씩 예측**할 수 있습니다.

In [17]:
import numpy as np

generated_ids = []
max_length = 250
for i in range(max_length):
    next_char = int(np.argmax(predictions, axis=-1)[0])
    generated_ids.append(next_char)
    inputs = keras.ops.expand_dims([next_char], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

모델이 예측한 결과를 확인하기 위해 출력된 **정수 시퀀스를 문자열로 변환**해 보겠습니다. 입력값을 토큰화 해제하려면 모든 토큰 ID를 문자열로 매핑하고 이를 결합하면 됩니다.

다음과 같은 출력이 나타납니다.

In [18]:
output = "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)


KING RICHARD III:
Call me the blood of noble father flower the death of the deed?

NORTHUMBERLAND:
My lords, for the good gracious lord,
I do not sleep all here at him. See where he would not
stay the triumph of the good work of her fresh.

GLOUCESTER:
And I, that I m


아직 다음 대비극을 만들어내지는 못했지만, 최소한의 데이터셋으로 2분 정도 학습시킨 결과치고는 나쁘지 않습니다. 이 간단한 예제의 목적은 언어 모델 설정의 강력함을 보여주는 것입니다. 우리는 한 번에 한 글자씩 추측하는 좁은 문제로 모델을 학습시켰지만, 이 모델을 훨씬 더 광범위한 문제, 즉 셰익스피어 작품처럼 끝없이 이어지는 텍스트 응답을 생성하는 데 활용했습니다.

이러한 학습 설정이 가능한 이유는 순환 신경망이 시퀀스에서 정보를 앞으로만 전달하기 때문이라는 점에 유의해야 합니다. 원한다면 GRU 레이어를 Bidirectional(GRU(...))으로 바꿔보세요. 학습 정확도가 즉시 99%를 넘어설 것이고, 생성 기능은 완전히 작동을 멈출 것입니다. 학습 과정에서 우리 모델은 매 학습 단계마다 전체 시퀀스를 접합니다. 만약 시퀀스에서 다음 토큰의 정보가 현재 토큰의 예측에 영향을 미치도록 "꼼수"를 부린다면, 문제는 아주 쉬워지는 것입니다.

이러한 언어 모델링 설정은 텍스트 영역의 수많은 문제에 대한 기본 토대가 됩니다. 또한 이 책에서 지금까지 살펴본 다른 모델링 문제들과 비교했을 때 다소 독특한 측면도 있습니다. 단순히 `model.predict()`를 호출하는 것만으로는 원하는 출력을 얻을 수 없습니다. 추론 시점에만 존재하는 복잡한 루프와 상당한 양의 로직이 있기 때문입니다! RNN 셀의 상태 순환은 학습과 추론 모두에서 발생하지만, 학습 과정에서는 모델이 예측한 레이블을 다시 입력으로 사용하는 경우는 없습니다.

### Sequence-to-sequence learning

언어 모델 개념을 확장하여 중요한 문제인 **기계 번역**을 다뤄 보겠습니다. **번역은 시퀀스-투-시퀀스 모델링(seq2seq)이라고 불리는 모델링 문제**의 한 유형입니다. **원문을 고정된 입력 시퀀스로 받아 번역된 텍스트 시퀀스를 결과로 생성하는 모델**을 구축하는 것이 목표입니다. **질의응답 또한 대표적인 시퀀스-투-시퀀스 문제**입니다.

시퀀스-투-시퀀스 모델의 **기본적인 구조**는 그림 15.1에 나와 있습니다. **학습 과정**에서는 다음과 같은 단계가 진행됩니다.

* **인코더** 모델은 **원문 시퀀스를 중간 표현으로 변환**합니다.
* **디코더**는 앞서 살펴본 언어 모델링 방식을 사용하여 학습됩니다. 디코더는 **이전의 모든 대상(target) 토큰과 인코더가 생성한 원문 시퀀스의 표현을 이용하여 대상(target) 시퀀스의 다음 토큰을 재귀적으로 예측**합니다.

**추론 단계**에서는 대상(target) 시퀀스에 접근할 수 없습니다. 처음부터 대상 시퀀스를 예측해야 합니다. 셰익스피어 생성기에서 했던 것처럼 토큰을 하나씩 순차적으로 생성해 보겠습니다.

* **인코더에서 인코딩된 소스 시퀀스를 얻습**니다.
* **디코더는 인코딩된 소스 시퀀스와 초기 "시드(seed)" 토큰(예: "[start]" 문자열)을 사용하여 시퀀스의 첫 번째 실제 토큰을 예측**합니다.
* **지금까지 예측된 시퀀스는 디코더에 반복적으로 입력되어 멈춤(stop) 토큰(예: "[end]" 문자열)이 생성될 때까지 계속**됩니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/seq2seq-learning.0e1e1c31.png" width="600"><br>Figure 15.1: Sequence-to-sequence learning: the source sequence is processed by the encoder and is then sent to the decoder. The decoder looks at the target sequence so far and predicts the target sequence offset by one step in the future. During inference, we generate one target token at a time and feed it back into the decoder.</p>

<p style="text-align:center">
<img src="rnn2.png" width="600"><br>Figure 15.1-1:</p>

시퀀스-투-시퀀스 번역 모델을 구축해 봅시다.

#### English-to-Spanish translation

우리는 **영어-스페인어 번역 데이터셋**을 사용할 것입니다. 다운로드해 봅시다.

In [19]:
import pathlib

zip_path = keras.utils.get_file(
    origin=(
        "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    ),
    fname="spa-eng",
    extract=True,
)
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [20]:
text_path

PosixPath('/root/.keras/datasets/spa-eng/spa-eng/spa.txt')

텍스트 파일에는 **각 줄마다 하나의 예시**가 있습니다. **영어 문장** 다음에 **탭 문자**가 오고, 그 다음에 해당 **스페인어 문장**이 옵니다. 이 파일을 분석해 보겠습니다.

In [21]:
with open(text_path, encoding="utf-8") as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start] " + spanish + " [end]"
    text_pairs.append((english, spanish))

저희의 **텍스트 쌍**은 다음과 같습니다.

In [22]:
import random
random.choice(text_pairs)

("Tom isn't a man of many words.",
 '[start] Tom no es un hombre de muchas palabras. [end]')

이제 이 데이터셋들을 섞어서 일반적인 **학습, 검증, 테스트 세트**로 나누어 보겠습니다.

In [23]:
import random

random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

다음으로, **영어와 스페인어 각각에 대한 두 개의 별도 텍스트 벡터화 레이어**를 준비해 보겠습니다. 문자열 전처리 방식을 사용자 정의해야 합니다.

* 삽입한 **"[start]"와 "[end]" 토큰을 유지해야** 합니다. 기본적으로 [ ] 문자는 제거되지만, "start"라는 단어와 시작 토큰 "[start]"를 구분하기 위해 이 문자들을 남겨두어야 합니다.
* 언어마다 구두점 표기법이 다릅니다! **스페인어 텍스트 벡터화 레이어에서 구두점을 제거**하려면 **¿ 문자도 함께 제거**해야 합니다.

참고로, **실제 번역 모델**에서는 구두점을 제거하는 대신 별도의 토큰으로 처리하여 **구두점이 있는 문장도 생성할 수 있도록** 해야 합니다. 하지만 여기서는 간단하게 모든 구두점을 제거하겠습니다.

In [24]:
import string
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )

vocab_size = 15000
sequence_length = 20

english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

마지막으로, **데이터를 `tf.data` 파이프라인으로 변환**할 수 있습니다. 이 파이프라인은 `inputs`와 `spanish` 두 개의 키를 가진 딕셔너리 `(inputs, target, sample_weights)` 튜플을 반환하도록 설계되었습니다. 
* **`inputs`는 토큰화된 영어 문장 `english`와 `spanish`를 포함하는 딕셔너리**이고,
* **`target`은 한 단계 앞선 스페인어 문장의 오프셋 값**입니다.
* **`sample_weights`는 Keras에게 손실과 메트릭을 계산할 때 사용할 레이블을 지정**하는 데 사용됩니다. 출력 번역문의 길이는 모두 같지 않으며, 일부 레이블 시퀀스는 0으로 채워집니다. 우리는 실제 번역된 텍스트를 나타내는 0이 아닌 레이블에 대한 예측만 중요하게 생각합니다.

이는 방금 구축한 생성 모델에서 설정한 "오프 바이 원(off by one)" 레이블과 동일하며, 고정된 인코더 입력이 추가된 것입니다. 인코더 입력은 모델에서 별도로 처리됩니다.

In [25]:
batch_size = 64

def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = spanish_tokenizer(spa)
    features = {"english": eng, "spanish": spa[:, :-1]}
    labels = spa[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

다음은 저희 데이터셋 출력 결과입니다.

In [26]:
inputs, targets, sample_weights = next(iter(train_ds))
print(inputs["english"].shape)

(64, 20)


In [27]:
print(inputs["spanish"].shape)

(64, 20)


In [28]:
print(targets.shape)

(64, 20)


In [29]:
print(sample_weights.shape)

(64, 20)


이제 데이터가 준비되었으니, 모델을 구축할 차례입니다.

#### Sequence-to-sequence learning with RNNs

앞서 언급한 트윈 인코더/디코더 설정을 시도하기 전에 더 간단한 옵션부터 살펴보겠습니다. RNN을 사용하여 **하나의 시퀀스를 다른 시퀀스로 변환**하는 가장 쉽고 단순한 방법은 **각 시간 단계에서 RNN의 출력을 유지**하고 이를 기반으로 출력 토큰을 예측하는 것입니다. Keras에서는 다음과 같이 표현할 수 있습니다.
```
inputs = keras.Input(shape=(sequence_length,), dtype="int32")
x = layers.Embedding(input_dim=vocab_size, output_dim=128)(inputs)
x = layers.LSTM(32, return_sequences=True)(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)
```

<p style="text-align:center">
<img src="https://camo.githubusercontent.com/e62f423b9d1858652d1a63f88b817f83c7572f4e298c8f9a1d59f9bc36e45380/68747470733a2f2f6d69726f2e6d656469756d2e636f6d2f6d61782f313430302f312a7a654f473930746d796536353466516a4c736c6b51412e706e67" width="400"><br>Figure 15.1-2:</p>

하지만 이 접근 방식에는 **치명적인 문제**가 있습니다. RNN은 단계적으로 작동하기 때문에 소스 시퀀스의 0번째부터 N번째 토큰만 사용하여 타겟 시퀀스의 N번째 토큰을 예측합니다. "I will bring the bag to you."라는 문장을 스페인어로 번역한다고 가정해 보겠습니다. 스페인어로는 "Te traeré la bolsa"가 되는데, **번역의 첫 단어인 "Te"는 영어 원문의 "you"에 해당**합니다. **원문의 마지막 단어를 보지 않고는 번역의 첫 단어를 출력할 방법이 없습니다!**

**인간 번역가라면 번역을 시작하기 전에 원문 전체를 읽을 것**입니다. 특히 단어 순서가 매우 다른 언어를 다룰 때는 더욱 중요합니다. 그리고 이것이 바로 표준 시퀀스-투-시퀀스 모델이 하는 일입니다. **적절한 시퀀스-투-시퀀스 구성**(그림 15.2 참조)에서는 **먼저 인코더 RNN을 사용하여 전체 소스 시퀀스를 소스 텍스트의 단일 표현으로 변환**합니다. 이는 **RNN의 최종 출력**일 수도 있고, 또는 **최종 내부 상태 벡터**일 수도 있습니다. 우리는 **이 표현을 언어 모델 설정에서 디코더 RNN의 초기 상태로 사용**할 수 있습니다. 셰익스피어 생성기에서 사용했던 것처럼 초기 상태를 0으로 설정하는 대신 말이죠. 이 **디코더는 초기 RNN 상태에서 얻은 영어 시퀀스에 대한 모든 정보를 바탕으로 현재 번역어가 주어졌을 때 스페인어 번역어의 다음 단어를 예측하도록 학습**합니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/seq2seq-rnn.ec377d3b.png" width="600"><br>Figure 15.2: A sequence-to-sequence RNN: an RNN encoder is used to produce a vector that encodes the entire source sequence, which is used as the initial state for an RNN decoder.</p>

**GRU 기반 인코더와 디코더를 사용하여 Keras로 구현**해 보겠습니다. **먼저 인코더부터 시작**해 보죠. 인코더 시퀀스에서는 실제로 토큰을 예측하지 않으므로, 모델이 시퀀스 끝부분의 정보를 시작 부분으로 전달하는 방식인 "꼼수"를 걱정할 필요가 없습니다. 오히려 이렇게 하는 것이 좋습니다. 소스 시퀀스를 풍부하게 표현하고 싶기 때문입니다. **양방향 레이어를 사용하면 이를 구현**할 수 있습니다.

In [30]:
embed_dim = 256
hidden_dim = 1024

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
rnn_layer = layers.GRU(hidden_dim)
rnn_layer = layers.Bidirectional(rnn_layer, merge_mode="sum")
encoder_output = rnn_layer(x)

다음으로 디코더를 추가해 보겠습니다. **디코더는 인코딩된 원문 문장을 초기 상태로 받는 간단한 GRU 레이어**입니다. **그 위에 각 출력 단계에서 스페인어 어휘에 대한 확률 분포를 생성하는 Dense 레이어를 추가**합니다. 여기서는 이전 내용만을 기반으로 다음 토큰을 예측해야 하므로 양방향 RNN을 사용하면 손실 함수가 너무 단순해져 학습이 제대로 이루어지지 않을 수 있습니다.

In [31]:
target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)
rnn_layer = layers.GRU(hidden_dim, return_sequences=True)
x = rnn_layer(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn = keras.Model([source, target], target_predictions)

seq2seq 모델 전체를 살펴보겠습니다.

In [32]:
seq2seq_rnn.summary(line_length=80)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_2           │ (None, None, 256) │   3,840,000 │ english[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal (NotEqual)  │ (None, None)      │           0 │ english[0][0]      │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_3           │ (None, None, 256) │   3,840,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ bidirectional         │ (None, 1024)      │   7,876,608 │ embedding_2[0][0], │
│ (Bidirectional)       │                   │             │ not_equal[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ gru_3 (GRU)           │ (None, None,      │   3,938,304 │ embedding_3[0][0], │
│                       │ 1024)             │             │ bidirectional[0][… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_1 (Dropout)   │ (None, None,      │           0 │ gru_3[0][0]        │
│                       │ 1024)             │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_2 (Dense)       │ (None, None,      │  15,375,000 │ dropout_1[0][0]    │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 34,869,912 (133.02 MB)

 Trainable params: 34,869,912 (133.02 MB)

 Non-trainable params: 0 (0.00 B)

모델과 데이터가 모두 준비되었습니다. 이제 번역 모델 학습을 시작할 수 있습니다.

In [33]:
seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 134s 98ms/step - accuracy: 0.3563 - loss: 3.6814 - val_accuracy: 0.5021 - val_loss: 2.4706
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 121s 93ms/step - accuracy: 0.5369 - loss: 2.2509 - val_accuracy: 0.5970 - val_loss: 1.8804
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 122s 93ms/step - accuracy: 0.6237 - loss: 1.6260 - val_accuracy: 0.6313 - val_loss: 1.6827
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 121s 93ms/step - accuracy: 0.6814 - loss: 1.2420 - val_accuracy: 0.6439 - val_loss: 1.6362
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 122s 93ms/step - accuracy: 0.7278 - loss: 0.9896 - val_accuracy: 0.6509 - val_loss: 1.6177
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 122s 93ms/step - accuracy: 0.7629 - loss: 0.8243 - val_accuracy: 0.6576 - val_loss: 1.6339
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 122s 93ms/step - accuracy: 0.7852 - loss: 0.7242 - val_accuracy: 0.6580 - val_loss: 1.6727
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 121s 93ms/step - accuracy: 

우리는 학습 과정에서 검증 세트 성능을 모니터링하는 간단한 방법으로 정확도를 선택했습니다. **65%의 정확도**를 얻었는데, 이는 모델이 스페인어 문장에서 다음 단어를 평균 65%의 확률로 정확하게 예측한다는 의미입니다. **하지만 실제로는 다음 토큰 정확도가 기계 번역 모델에 적합한 지표는 아닙니다**. 특히, 토큰 N+1을 예측할 때 0부터 N까지의 올바른 목표 토큰을 이미 알고 있다는 가정을 전제로 하기 때문입니다. 실제 추론 과정에서는 목표 문장을 처음부터 생성해야 하므로 이전에 생성된 토큰이 100% 정확하다고 가정할 수 없습니다. 실제 기계 번역 시스템을 개발할 때는 더욱 신중하게 지표를 설계해야 합니다. **BLEU 점수와 같은 표준 지표는 기계 번역된 텍스트와 고품질 참조 번역 세트 간의 유사도를 측정하며, 약간의 순서 불일치를 허용**할 수 있습니다.

마지막으로, 우리의 모델을 사용하여 **추론을 수행**해 보겠습니다. 테스트 세트에서 몇 개의 문장을 선택하여 모델이 어떻게 번역하는지 확인해 보겠습니다. 
* 먼저 **시드 토큰인 "[start]"를 인코딩된 영어 원문과 함께 디코더 모델에 입력**합니다.
* **다음 토큰 예측값을 얻고, 이를 디코더에 반복적으로 다시 입력하면서 매 반복마다 새로운 목표 토큰을 하나씩 추출**합니다.
* 이 과정은 **"[end]"에 도달하거나 최대 문장 길이에 이를 때까지 계속**됩니다.

최종 모델 가중치는 가중치의 무작위 초기화와 입력 데이터의 무작위 섞기에 따라 달라지므로 정확한 번역 결과는 실행할 때마다 다를 수 있습니다. 결과는 다음과 같습니다.

In [34]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
He went to America to study medicine.
[start] Él fue a américa a estudiar medicina [end]
-
Are you seriously thinking about getting married again at your age?
[start] estás pensando seriamente en [UNK] a tu edad [end]
-
Country life is healthier than city life.
[start] la vida es más grande que la ciudad [end]
-
He accumulated a large fortune.
[start] Él acumuló una gran fortuna [end]
-
The train pulled into the station.
[start] el tren partió a la estación [end]


저희 모델은 기본적인 오류를 많이 범하지만, **장난감 모델치고는 꽤 잘 작동**합니다.

이 추론 방식은 매우 간단하지만, **새로운 단어를 샘플링할 때마다 전체 소스 문장과 생성된 전체 목표 문장을 다시 처리하기 때문에 비효율적**입니다. 실제 응용 프로그램에서는 변경되지 않은 상태를 다시 계산하지 않도록 주의해야 합니다. 디코더에서 새로운 토큰을 예측하는 데 필요한 것은 현재 토큰과 이전 RNN 상태뿐이며, 이는 각 반복 전에 캐시할 수 있습니다.

이 장난감 모델을 **개선할 수 있는 방법**은 많습니다. **인코더와 디코더 모두에 깊은 순환 레이어 스택을 사용**하거나, **LSTM과 같은 다른 RNN 레이어를 시도**해 볼 수도 있습니다. 하지만 이러한 수정 외에도 시퀀스-투-시퀀스 학습에 대한 **RNN 접근 방식에는 몇 가지 근본적인 한계**가 있습니다.

* 소스 시퀀스 표현은 인코더 상태 벡터에 완전히 저장되어야 하므로 **번역할 수 있는 문장의 크기와 복잡성이 크게 제한**됩니다.
* **RNN은 과거 정보를 점진적으로 잊어버리는 경향**이 있어 **매우 긴 시퀀스를 처리하는 데 어려움**을 겪습니다. 시퀀스의 100번째 토큰에 도달할 때쯤이면 시퀀스 시작 부분에 대한 정보가 거의 남아 있지 않습니다.

순환 신경망(RNN)은 2010년대 중반 시퀀스-투-시퀀스 학습을 지배했습니다. **2017년경의 Google 번역**은 방금 만든 것과 유사한 구조로 **7개의 대형 LSTM 레이어를 쌓아서 구동**되었습니다. 그러나 이러한 RNN의 한계로 인해 연구자들은 결국 트랜스포머(Transformer)라고 불리는 새로운 유형의 시퀀스 모델을 개발하게 되었습니다.

### The Transformer architecture

2017년, Vaswani 외 연구진은 획기적인 논문 **"Attention Is All You Need"[1]에서 Transformer 아키텍처를 소개**했습니다. 저자들은 우리가 방금 구축한 것과 같은 번역 시스템을 연구하고 있었는데, 핵심적인 발견은 논문 제목에 담겨 있습니다. 바로 **어텐션(attention)이라는 간단한 메커니즘을 사용하여 순환 레이어(recurrent layer)를 전혀 사용하지 않고도 강력한 시퀀스 모델을 구축할 수 있다는 것**입니다. 어텐션 개념 자체는 새로운 것이 아니었고, 논문 발표 당시 이미 몇 년 동안 자연어 처리 시스템에서 사용되고 있었습니다. 하지만 **어텐션이 시퀀스를 통해 정보를 전달하는 데 필요한 유일한 메커니즘이 될 수 있을 정도로 유용하다는 사실은 당시로서는 매우 놀라운 발견**이었습니다.

이 발견은 자연어 처리 분야는 물론 그 너머까지 혁명적인 변화를 가져왔습니다. 어텐션은 딥러닝에서 가장 영향력 있는 개념 중 하나로 빠르게 자리 잡았습니다. 이 섹션에서는 **어텐션의 작동 방식과 시퀀스 모델링에 왜 그렇게 효과적인지 자세히 설명**합니다. 그런 다음 **어텐션을 사용하여 영어-스페인어 번역 모델을 다시 구축**해 보겠습니다.

그렇다면, 지금까지 설명한 내용을 바탕으로 **어텐션이란** 정확히 무엇일까요? 그렇다면 어텐션은 지금까지 사용해 온 순환 신경망(RNN)을 어떻게 대체할 수 있을까요?

어텐션은 사실 방금 만든 RNN 모델과 같은 기존 RNN 모델을 강화하기 위해 개발되었습니다. 연구자들은 **RNN이 주변 영역 내의 의존성을 모델링하는 데는 탁월**하지만, **시퀀스 길이가 길어질수록 재현율이 떨어지는 것을 발견**했습니다. 예를 들어, 문서에 대한 질문에 답하는 시스템을 구축한다고 가정해 보겠습니다. 문서 길이가 너무 길어지면 RNN의 결과는 인간의 예측 능력과는 비교할 수 없을 정도로 형편없어집니다.

이 책을 활용하여 날씨 예측 모델을 만든다고 생각해 보세요. 시간이 충분하다면 책 전체를 처음부터 끝까지 읽겠지만, 실제로 모델을 구현할 때는 시계열 관련 장에 특히 집중할 것입니다. 같은 장 안에서도 자주 참고할 특정 코드 예제나 설명이 있을 것입니다. 반면에 코드를 작성할 때는 이미지 컨볼루션과 같은 세부적인 내용에는 그다지 신경 쓰지 않을 것입니다. 이 책의 전체 단어 수는 10만 단어를 훨씬 넘는데, 이는 우리가 지금까지 다뤄본 어떤 시퀀스 길이보다 훨씬 깁니다. 하지만 인간은 텍스트에서 정보를 추출할 때 선택적이고 맥락적인 방식을 사용할 수 있습니다.

반면, **RNN은 시퀀스의 이전 부분을 직접 참조할 수 있는 메커니즘이 없습**니다. **모든 정보는 설계상 RNN 셀의 내부 상태를 거쳐 시퀀스의 모든 위치를 순환적으로 통과**해야 합니다. 마치 이 책을 다 읽고 덮은 다음, 날씨 예측 모델을 완전히 기억에 의존해서 구현하려는 것과 같습니다. **어텐션 메커니즘은 신경망이 현재 처리 중인 입력에 따라 시퀀스의 특정 부분에 더 많은 가중치를 부여하고 다른 부분에는 더 적은 가중치를 부여할 수 있도록 하는 메커니즘을 구축**하는 것입니다(그림 15.3).

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/attention-concept.fde57742.png" width="400"><br>Figure 15.3: The general concept of attention in deep learning: input features get assigned attention scores, which can be used to inform the next representation of the input.</p>


**아인슈타인 합 표기법이란 무엇일까요?**

머신러닝 코드에서 `np.einsum('ij,jk->ik', a, b)`와 같은 작은 코드 조각을 자주 볼 수 있습니다. 이것은 **아인슈타인 합 표기법(Einstein summation notation)의 줄임말**로, 아인슈타인 합 표기법이라고 합니다. 이 표기법을 익히면 복잡한 배열 연산을 명확하게 표현하는 데 유용하게 사용할 수 있습니다. 특히 Transformer 코드에서 자주 사용되는 이유입니다.

아인슈타인 합 표기법의 핵심 아이디어는 **입력의 각 축을 고유한 문자로 나타내는 것**입니다. 예를 들어, 3차 입력은 `ijk`로 표현할 수 있습니다. 그런 다음, 입력 개수는 상관없고 출력은 `input1,input2->output`과 같이 하나의 축을 갖는 표기법을 작성합니다. 이 표기법의 규칙은 다음과 같습니다.

* **입력에 동일한 문자가 있는 경우, 해당 축의 값을 서로 곱**합니다. 이때 두 축의 크기는 같아야 합니다.
* **입력에는 있지만 출력에는 없는 문자가 있는 경우, 해당 축의 값을 모두 더하여** 출력 배열에 나타나지 않도록 합니다.
* 출력 축은 어떤 순서로든 반환될 수 있습니다.


몇 가지 예시를 살펴보면 훨씬 더 명확해집니다.
```
# Transposes
np.einsum("ij->ji")
# matmul
np.einsum("ij,jk->ik")
# matmuls a list of matrices against a single matrix
np.einsum("hij,jk->hik")
# Dot-product
np.einsum("i,i->")
# Element-wise multiplication
np.einsum("ijk,ijk->ijk")
# Element-wise multiplies and sums everything.
np.einsum("ijk,ijk->")
```
Keras에서는 두 가지 방법으로 einsum을 사용할 수 있습니다. keras.ops.einsum은 np.einsum을 대체하는 함수이고, **keras.layers.EinsumDense는 matmul 연산 대신 einsum 연산을 사용하는 Dense 레이어**입니다.

#### Dot-product attention

번역 RNN을 다시 살펴보고 **선택적 어텐션 개념을 추가**해 보겠습니다. **단일 토큰을 예측하는 경우**를 생각해 봅시다. 소스 및 타겟 시퀀스를 GRU 레이어에 통과시키면 **예측하려는 타겟 토큰을 나타내는 벡터**와 **소스 텍스트의 각 단어를 나타내는 벡터 시퀀스**를 얻게 됩니다.

어텐션을 통해 모델이 현재 예측하려는 단어와의 관련성을 기준으로 소스 시퀀스의 모든 벡터에 점수를 매길 수 있도록 하는 것이 목표입니다(그림 15.4). **소스 토큰의 벡터 표현이 높은 점수를 받으면 특히 중요하다고 간주하고, 그렇지 않으면 덜 중요하게 여깁니다. 지금은 score(target_vector, source_vector)라는 함수가 있다고 가정**해 보겠습니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/attention.6007731a.png" width="400"><br>Figure 15.4: Attention assigns a relevance score to each vector in a source for each vector in a target sequence.</p>

어텐션 메커니즘이 제대로 작동하려면 **중요한 토큰에 대한 정보를 소스 및 타겟 시퀀스의 총 길이만큼 길어질 수 있는 루프를 통해 전달하는 것을 피해야** 합니다. 바로 이 지점에서 RNN이 한계를 드러내기 시작합니다. 이를 해결하는 간단한 방법은 **계산된 점수를 기반으로 모든 소스 벡터의 가중 합을 구하는 것**입니다. 또한 **특정 타겟에 대한 모든 어텐션 점수의 합이 1이면 가중 합이 예측 가능한 크기를 가지므로 편리**합니다. 이를 위해 점수에 **소프트맥스 함수를 적용**할 수 있습니다. NumPy 의사 코드로는 다음과 같습니다.
```
scores = [score(target, source) for source in sources]
scores = softmax(scores)
combined = np.sum(scores * sources)
```
하지만 이 **관련성 점수는 어떻게 계산해야 할까요?** 연구자들이 처음 어텐션 메커니즘을 다룰 때, 이 질문은 중요한 연구 주제였습니다. 가장 간단한 접근 방식 중 하나가 가장 효과적이라는 것이 밝혀졌습니다. **타겟 벡터와 소스 벡터 사이의 거리를 간단하게 측정하기 위해 내적(dot-product)을 사용**할 수 있습니다. 소스 벡터와 타겟 벡터가 서로 가까우면, 소스 토큰이 예측과 관련성이 높다고 가정합니다. 이 장의 마지막 부분에서 이러한 가정이 직관적으로 타당한 이유를 살펴보겠습니다.

이제 의사 코드를 업데이트해 보겠습니다. 전체 타겟 시퀀스를 한 번에 처리하도록 코드를 더 완벽하게 만들 수 있습니다. 이는 이전 코드를 타겟 시퀀스의 각 토큰에 대해 반복문으로 실행하는 것과 같습니다. **타겟과 소스 모두 시퀀스인 경우, 어텐션 점수는 행렬로 표현**됩니다. **각 행은 가중합에서 타겟 단어가 소스 단어에 부여하는 가치**를 나타냅니다(그림 15.5 참조). 내적과 가중합을 편리하게 표현하기 위해 Einsum 표기법을 사용하겠습니다.

```
def dot_product_attention(target, source):
    # Takes the dot-product between all target and source vectors,
    # where b = batch size, t = target length, s = source length, and d
    # = vector size
    scores = np.einsum("btd,bsd->bts", target, source)
    scores = softmax(scores, axis=-1)
    # Computes a weighted sum of all source vectors for each target
    # vector
    return np.einsum("bts,bsd->btd", scores, source)

dot_product_attention(target, source)
```
<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/attention-scores.2932e0ff.png" width="300"><br>Figure 15.5: When both target and source are sequences, attention scores are a 2D matrix. Each row shows the attention scores for the word we are trying to predict (in green).</p>

어텐션 메커니즘의 가설 공간을 훨씬 풍부하게 만들려면 모델에 어텐션 점수를 제어하는 ​​매개변수를 제공해야 합니다. **소스 벡터와 타겟 벡터를 모두 Dense 레이어로 투영하면, 모델은 전반적인 예측 품질 향상에 도움이 되는 소스 벡터와 타겟 벡터가 가까운 최적의 공유 공간을 찾을 수 있습**니다. 마찬가지로, **소스 벡터를 결합하기 전과 합산 후 완전히 별개의 공간으로 투영할 수 있도록 해야** 합니다.

또한, 업계에서 표준으로 자리 잡은 입력 이름 체계를 약간 다르게 사용할 수 있습니다. 방금 작성한 코드는 대략 **sum(score(target, source) * source)로 요약**할 수 있습니다. 이를 입력 이름을 다르게 하여 **sum(score(query, key) * value)와 같이 표현**할 수도 있습니다. 이 세 개의 인자를 사용하는 버전은 더 일반적입니다. 드물지만 소스 입력의 점수를 매기는 데 사용하는 벡터와 소스 입력을 합산하는 데 사용하는 벡터가 다를 수 있기 때문입니다.

이러한 용어는 검색 엔진과 추천 시스템에서 유래했습니다. 데이터베이스에서 사진을 검색하는 검색 도구를 상상해 보세요. 여기서 "쿼리(query)"는 검색어이고, "키(keys)"는 쿼리와 일치하는 사진 태그이며, 마지막으로 "값(values)"은 사진 자체입니다(그림 15.6). 우리가 구축하고 있는 어텐션 메커니즘은 이러한 종류의 검색과 대략적으로 유사합니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/query-key-value.b57cceb0.png" width="500"><br>Figure 15.6: Retrieving images from a database: the query is compared to a set of keys, and the match scores are used to rank values (images).</p>

이제 새로운 용어를 사용하여 매개변수화된 어텐션을 표현하도록 의사 코드를 업데이트해 보겠습니다.

```
query_dense = layers.Dense(dim)
key_dense = layers.Dense(dim)
value_dense = layers.Dense(dim)
output_dense = layers.Dense(dim)

def parameterized_attention(query, key, value):
    query = query_dense(query)
    key = key_dense(key)
    value = value_dense(value)
    scores = np.einsum("btd,bsd->bts", query, key)
    scores = softmax(scores, axis=-1)
    outputs = np.einsum("bts,bsd->btd", scores, value)
    return output_dense(outputs)

parameterized_attention(query=target, key=source, value=source)
```
이 블록은 완벽하게 작동하는 어텐션 메커니즘입니다! 방금 작성한 함수는 디코딩하려는 목표 단어에 따라 소스 시퀀스의 어느 위치에서든 문맥에 맞는 정보를 가져올 수 있도록 합니다.

"어텐션이 전부다"의 저자들은 시행착오를 통해 우리 메커니즘에 두 가지를 더 수정했습니다. 첫 번째는 간단한 스케일링 계수입니다. **입력 벡터가 길어지면 내적 점수가 상당히 커질 수** 있는데, 이는 소프트맥스 기울기의 안정성에 영향을 미칠 수 있습니다. 해결책은 간단합니다. 소프트맥스 점수를 약간 줄이면 됩니다. **벡터 길이의 제곱근으로 스케일링하면 어떤 벡터 크기에도 잘 작동**합니다.

두 번째는 어텐션 메커니즘의 표현력과 관련이 있습니다. 우리가 사용하는 소프트맥스 합은 강력합니다. 시퀀스의 멀리 떨어진 부분들을 직접 연결할 수 있게 해주기 때문입니다. 하지만 이 합산 방식은 다소 투박합니다. **모델이 한 번에 너무 많은 토큰에 주의를 기울이려고 하면 개별 소스 토큰의 흥미로운 특징들이 결합된 표현에서 "희석"될 수** 있습니다. 효과적인 간단한 방법은 **동일한 시퀀스에 대해 여러 개의 서로 다른 어텐션 헤드를 사용**하여 서로 다른 매개변수로 동일한 계산을 수행함으로써 이 어텐션 연산을 여러 번 수행하는 것입니다.

```
query_dense = [layers.Dense(head_dim) for i in range(num_heads)]
key_dense = [layers.Dense(head_dim) for i in range(num_heads)]
value_dense = [layers.Dense(head_dim) for i in range(num_heads)]
output_dense = layers.Dense(head_dim * num_heads)

def multi_head_attention(query, key, value):
    head_outputs = []
    for i in range(num_heads):
        query = query_dense[i](query)
        key = key_dense[i](key)
        value = value_dense[i](value)
        scores = np.einsum("btd,bsd->bts", target, source)
        scores = softmax(scores / math.sqrt(head_dim), axis=-1)
        head_output = np.einsum("bts,bsd->btd", scores, source)
        head_outputs.append(head_output)
    outputs = ops.concatenate(head_outputs, axis=-1)
    return output_dense(outputs)

multi_head_attention(query=target, key=source, value=source)
```
쿼리와 키를 서로 다르게 투영함으로써, **하나의 헤드는 소스 문장의 주어와 일치하도록 학습**하고, **다른 헤드는 구두점에 주의를 기울일 수** 있습니다. 이러한 다중 헤드 어텐션은 전체 소스 시퀀스를 단일 소프트맥스 합으로 결합해야 하는 제약을 피할 수 있습니다(그림 15.7).
<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/multi-head-attention.718456ad.png" width="400"><br>Figure 15.7: Multi-headed attention allows each target word to attend to different parts of the source sequence in separate partitions of the eventual output vector.</p>

물론 실제로는 이 코드를 재사용 가능한 레이어로 작성하는 것이 좋습니다. 케라스는 이러한 기능을 제공합니다. **MultiHeadAttention 레이어**를 사용하면 이전 코드를 다음과 같이 다시 생성할 수 있습니다.

```
multi_head_attention = keras.layers.MultiHeadAttention(
    num_heads=num_heads,
    head_dim=head_dim,
)
multi_head_attention(query=target, key=source, value=source)
```

#### Transformer encoder block

**MultiHeadAttention 레이어**를 사용하는 한 가지 방법은 **기존 RNN 번역 모델에 추가**하는 것입니다. **인코더와 디코더의 시퀀스 출력을 어텐션 레이어로 전달**하고, 그 **출력을 사용하여 예측 전에 목표 시퀀스를 업데이트**할 수 있습니다. **어텐션은 GRU 레이어가 처리하기 어려운 텍스트 내의 장거리 의존성을 모델이 처리할 수 있도록** 해줍니다. 실제로 이는 **RNN 모델의 성능을 향상**시키며, 2010년대 중반에 어텐션이 처음 사용된 방식입니다.

하지만 "Attention is all you need"의 저자들은 **어텐션을 더 나아가 모델 내 모든 시퀀스 데이터를 처리하는 일반적인 메커니즘으로 사용할 수 있다는 점**을 깨달았습니다. 지금까지는 두 시퀀스 간의 정보 전달을 처리하는 방법으로만 어텐션을 살펴보았지만, **시퀀스가 ​​자기 자신에게 어텐션하도록 하는 방법으로도 어텐션을 사용**할 수 있습니다.

```
multi_head_attention(key=source, value=source, query=source)
```
이것을 **셀프 어텐션(self-attention**)이라고 하며, 매우 강력한 기능입니다. 셀프 어텐션을 사용하면 **각 토큰이 자기 자신을 포함하여 해당 시퀀스 내의 모든 토큰에 주의를 기울일 수 있으므로, 모델은 문맥 속에서 단어를 표현하는 방법을 학습**할 수 있습니다.

예를 들어 **"The train left the station on time.**"라는 문장을 생각해 보겠습니다. 이제 문장에서 "station"이라는 단어를 생각해 보세요. 어떤 종류의 역을 말하는 걸까요? 라디오 방송국일까요? 아니면 국제 우주 정거장일까요? **셀프 어텐션을 사용하면 모델은 "station"과 "train"이라는 두 단어 쌍에 높은 어텐션 점수를 부여**하도록 학습할 수 있으며, **"train"을 표현하는 데 사용되는 벡터를 "station"이라는 단어를 표현하는 벡터에 더할 수** 있습니다.

**셀프 어텐션은 모델이 단어를 독립적으로 표현하는 것에서 나아가 시퀀스에 나타나는 다른 모든 토큰을 고려하여 단어를 표현하는 효과적인 방법을 제공**합니다. 이는 RNN이 하는 일과 매우 유사해 보입니다. 그렇다면 RNN 레이어를 MultiHeadAttention으로 대체할 수 있을까요?

거의 그렇습니다! 하지만 완전히 그렇지는 않습니다. 모든 심층 신경망에 필수적인 요소인 **비선형 활성화 함수가 여전히 필요**합니다. **MultiHeadAttention 레이어는 소스 시퀀스의 모든 요소에 대한 선형 투영을 결합**하지만, 그게 전부입니다. 어떻게 보면, **매우 표현력이 풍부한 풀링 연산**이라고 할 수 있습니다. **극단적인 경우를 생각해 보면, 토큰 길이가 1인 경우입니다. 이 경우, 어텐션 스코어 행렬은 항상 단일 행렬이 되며, 전체 레이어는 비선형성 없이 소스 시퀀스의 선형 투영으로 축소**됩니다. 어텐션 레이어를 100개 쌓아도 전체 계산을 단 하나의 행렬 곱셈으로 단순화할 수 있습니다! 이것이 바로 우리 모델의 표현력에 대한 실제적인 문제입니다.

어느 시점에서 모든 순환 셀은 각 토큰에 대한 입력 벡터를 밀집 투영을 통해 전달하고 활성화 함수를 적용합니다. 우리도 이와 유사한 과정을 구현해야 합니다. "어텐션이 전부다(Attention is all you need)"의 저자들은 이를 가능한 한 가장 간단한 방법으로 다시 추가하기로 결정했습니다. 바로 **두 개의 밀집 레이어 사이에 활성화 함수를 두고 피드포워드 네트워크를 쌓는 것**입니다. **어텐션은 시퀀스 전체에 정보를 전달하고, 피드포워드 네트워크는 개별 시퀀스 항목의 표현을 업데이트**합니다.

이제 Transformer 모델 구축을 시작할 준비가 되었습니다. 먼저 번역 **모델의 인코더를 교체**해 보겠습니다. 영어 단어로 이루어진 원문 시퀀스를 따라 정보를 전달하기 위해 셀프 어텐션을 사용할 것입니다. 또한 9장에서 컨볼루션 신경망을 구축할 때 특히 중요하다고 배웠던 두 가지 요소, 즉 **정규화(normalization)와 잔차 연결(residual connections)도 추가**할 것입니다.

In [35]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        residual = x = source
        mask = source_mask[:, None, :]
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

여기서 사용하는 정규화 레이어는 이미지 모델에서 사용했던 **배치 정규화(BatchNormalization) 레이어가 아니라는 점에 유의**하세요. 배치 정규화는 시퀀스 데이터에 적합하지 않기 때문입니다. 대신, **각 시퀀스를 배치 내의 다른 시퀀스와 독립적으로 정규화하는 레이어 정규화(LayerNormalization) 레이어를 사용**합니다. 다음은 NumPy 스타일의 의사 코드입니다.

```
# Input shape: (batch_size, sequence_length, embedding_dim)
def layer_normalization(batch_of_sequences):
    # To compute mean and variance, we only pool data over the last
    # axis.
    mean = np.mean(batch_of_sequences, keepdims=True, axis=-1)
    variance = np.var(batch_of_sequences, keepdims=True, axis=-1)
    return (batch_of_sequences - mean) / variance
```
배치 정규화(학습 중)와 비교해 보세요:
```
# Input shape: (batch_size, height, width, channels)
def batch_normalization(batch_of_images):
    # Pools data over the batch axis (axis 0), which creates
    # interactions between samples in a batch
    mean = np.mean(batch_of_images, keepdims=True, axis=(0, 1, 2))
    variance = np.var(batch_of_images, keepdims=True, axis=(0, 1, 2))
    return (batch_of_images - mean) / variance
```
배치 정규화(BatchNormalization)는 여러 샘플의 정보를 수집하여 특징 평균과 분산에 대한 정확한 통계를 얻는 반면, **레이어 정규화(LayerNormalization)는 각 시퀀스 내의 데이터를 개별적으로 풀링하므로 시퀀스 데이터에 더 적합**합니다.

또한, **멀티헤드 어텐션(MultiHeadAttention) 레이어에 attention_mask라는 새로운 입력을 전달**합니다. 이 불리언 텐서 입력은 어텐션 스코어와 동일한 형태(batch_size, target_length, source_length)로 브로드캐스트됩니다. **attention_mask가 설정되면 특정 위치의 어텐션 스코어가 0이 되어 해당 위치의 소스 토큰이 어텐션 계산에 사용되지 않게 됩니다.** 이는 시퀀스 내의 어떤 토큰도 정보가 없는 패딩 토큰에 어텐션하지 않도록 하기 위함입니다. **인코더 레이어는 입력에서 패딩 토큰이 아닌 모든 토큰을 표시하는 source_mask 입력을 받아 (batch_size, 1, source_length) 형태로 업랭크하여 attention_mask로 사용**합니다.

이 레이어의 입력과 출력은 동일한 형태를 가지므로 인코더 블록을 서로 쌓아 올려 입력 영어 문장을 점진적으로 더 표현력 있게 표현할 수 있다는 점에 유의하십시오.

In [36]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, target, source, source_mask):
        residual = x = target
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        mask = source_mask[:, None, :]
        x = self.cross_attention(
            query=x, key=source, value=source, attention_mask=mask
        )
        x = x + residual
        x = self.cross_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

**디코더 레이어는 타겟과 소스 모두를 입력**으로 받습니다. TransformerEncoder와 마찬가지로, 소스 입력에서 패딩의 위치를 ​​표시하는 `source_mask`를 입력으로 받습니다(패딩이 없으면 True, 있으면 False). 이 **`source_mask`는 크로스 어텐션 레이어의 `attention_mask`로 사용**됩니다.

**디코더의 셀프 어텐션 레이어에는 다른 유형의 어텐션 마스크가 필요**합니다. RNN 디코더를 구축할 때 양방향 RNN을 사용하지 않았던 것을 기억하세요. 양방향 RNN을 사용했다면 모델이 예측하려는 레이블을 특징으로 인식하여 속임수를 쓸 수 있었기 때문입니다! 어텐션은 본질적으로 양방향입니다. 셀프 어텐션에서는 타겟 시퀀스의 어떤 토큰 위치든 다른 어떤 위치에도 어텐션을 적용할 수 있습니다. 특별한 조치를 취하지 않으면 모델은 시퀀스의 다음 토큰을 현재 레이블로 인식하고 새로운 번역을 생성하는 능력을 잃게 됩니다.

이러한 문제를 해결하기 위해 특별한 **"인과적(causal)" 어텐션 마스크를 사용하여 단방향 정보 흐름을 구현**할 수 있습니다. 예를 들어, 다음과 같이 하삼각 영역에 1이 포함된 어텐션 마스크를 입력으로 받는다고 가정해 보겠습니다.
```
[
    [1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0],
    [1, 1, 1, 1, 0],
    [1, 1, 1, 1, 1],
]
```
각 행 i는 위치 i에 있는 대상 토큰에 대한 어텐션 마스크로 해석될 수 있습니다. **첫 번째 행에서 첫 번째 토큰은 자신에게만 어텐션**할 수 있습니다. **두 번째 행에서 두 번째 토큰은 첫 번째와 두 번째 토큰 모두에 어텐션**할 수 있으며, 이러한 방식으로 계속됩니다. 이는 정보가 시퀀스에서 앞으로만 전파되고 뒤로는 전파되지 않는 RNN 레이어와 동일한 효과를 제공합니다. **Keras에서는 MultiHeadAttention 레이어를 호출할 때 use_casual_mask를 전달하여 이 하삼각(lower-triangular) 마스크를 지정**할 수 있습니다. 그림 15.8은 **Transformer 모델로 구성될 때 인코더 및 디코더 레이어의 시각적 표현**을 보여줍니다.

<p style="text-align:center">
<img src="https://deeplearningwithpython.io/images/ch15/encoder-decoder.d979dbbc.png" width="300"><br>Figure 15.8: A visual representation of the computations for both TransformerEncoder and TransformerDecoder blocks</p>

#### Sequence-to-sequence learning with a Transformer

이 모든 것을 종합해 보겠습니다. **RNN 모델과 동일한 기본 설정**을 사용하되, **GRU 레이어를 TransformerEncoder와 TransformerDecoder로 대체**합니다. 피드포워드 블록을 제외한 모델 전체에서 **임베딩 크기는 256**으로 유지합니다. **피드포워드 블록에서는 비선형 연산 전에 임베딩 크기를 2048로 확대**하고, **비선형 연산 후에는 다시 모델의 은닉층 크기로 축소**합니다. 이처럼 중간 차원을 크게 설정하는 것이 실제 구현에 효과적입니다.

In [37]:
hidden_dim = 256
intermediate_dim = 2048
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

트랜스포머 모델의 요약을 살펴보겠습니다.

In [38]:
transformer.summary(line_length=80)

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_4           │ (None, None, 256) │   3,840,000 │ english[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_2           │ (None, None)      │           0 │ english[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_encoder   │ (None, None, 256) │   1,315,072 │ embedding_4[0][0], │
│ (TransformerEncoder)  │                   │             │ not_equal_2[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_3           │ (None, None)      │           0 │ english[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding_5           │ (None, None, 256) │   3,840,000 │ spanish[0][0]      │
│ (Embedding)           │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_decoder   │ (None, None, 256) │   1,578,752 │ transformer_encod… │
│ (TransformerDecoder)  │                   │             │ not_equal_3[0][0], │
│                       │                   │             │ embedding_5[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_5 (Dropout)   │ (None, None, 256) │           0 │ transformer_decod… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_7 (Dense)       │ (None, None,      │   3,855,000 │ dropout_5[0][0]    │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 14,428,824 (55.04 MB)

 Trainable params: 14,428,824 (55.04 MB)

 Non-trainable params: 0 (0.00 B)

저희 모델은 이전에 학습시킨 GRU 번역 모델과 거의 동일한 구조를 가지고 있으며, **순환 레이어 대신 어텐션 메커니즘을 사용하여 시퀀스 전체에 정보를 전달**합니다. 이제 모델을 학습시켜 보겠습니다.

In [39]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 47s 30ms/step - accuracy: 0.3659 - loss: 1.4771 - val_accuracy: 0.5043 - val_loss: 1.0086
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 29s 23ms/step - accuracy: 0.5363 - loss: 0.9618 - val_accuracy: 0.5780 - val_loss: 0.8077
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 29s 23ms/step - accuracy: 0.6051 - loss: 0.7546 - val_accuracy: 0.6054 - val_loss: 0.7380
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.6486 - loss: 0.6307 - val_accuracy: 0.6163 - val_loss: 0.7098
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 29s 23ms/step - accuracy: 0.6817 - loss: 0.5448 - val_accuracy: 0.6260 - val_loss: 0.7028
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.7070 - loss: 0.4817 - val_accuracy: 0.6293 - val_loss: 0.7143
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.7290 - loss: 0.4330 - val_accuracy: 0.6334 - val_loss: 0.7181
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.7464 -

**학습 후 정확도는 약 58%에 도달**했습니다. 즉, 모델이 스페인어 문장에서 다음 단어를 평균 58%의 확률로 정확하게 예측한다는 뜻입니다. 뭔가 이상합니다. 학습 결과가 RNN 모델보다 7%포인트나 떨어집니다. 이 Transformer 아키텍처가 과대광고된 만큼 강력하지 않거나, 구현 과정에서 무언가 잘못된 부분이 있는 것 같습니다. 무엇이 문제인지 찾을 수 있나요?

이 부분은 표면적으로는 시퀀스 모델에 대한 내용입니다. 이전 장에서 단어 순서가 의미 전달에 얼마나 중요한지 살펴보았습니다. 하지만 **우리가 방금 구축한 Transformer는 사실 시퀀스 모델이 아닙니다**. 눈치채셨나요? 이 모델은 시퀀스 토큰을 서로 독립적으로 처리하는 완전 연결 레이어와 토큰들을 하나의 집합으로 인식하는 어텐션 레이어로 구성되어 있습니다. **시퀀스에서 토큰의 순서를 바꾸더라도 쌍별(pairwise) 어텐션 점수와 문맥 인식 표현은 동일하게 유지**됩니다. 모든 영어 원문 문장의 모든 단어 순서를 완전히 바꿔도 모델은 이를 알아채지 못하고 동일한 정확도를 유지합니다. 어텐션은 시퀀스 요소 쌍 간의 관계에 초점을 맞춘 집합 처리 메커니즘입니다. 즉, 이러한 요소가 시퀀스의 시작, 끝 또는 중간에 나타나는지 여부는 고려하지 않습니다. 그렇다면 왜 트랜스포머를 시퀀스 모델이라고 부를까요? 그리고 단어 순서를 고려하지 않는 트랜스포머가 어떻게 기계 번역에 적합할 수 있을까요?

RNN의 경우, 레이어의 연산이 순서를 인식하도록 설계되었습니다. 하지만 **트랜스포머의 경우, 임베딩된 시퀀스 자체에 위치 정보를 직접 삽입**합니다. 이를 **위치 임베딩(positional embedding**)이라고 합니다. 자세히 살펴보겠습니다.

#### Embedding positional information

위치 임베딩의 기본 아이디어는 매우 간단합니다. **모델이 단어 순서 정보를 활용할 수 있도록 각 단어 임베딩에 문장 내 단어의 위치를 ​​추가**하는 것입니다. 입력 단어 임베딩은 두 가지 구성 요소로 이루어져 있습니다. 
* 하나는 특정 문맥과 관계없이 단어를 나타내는 일반적인 **단어 벡터**이고,
* 다른 하나는 현재 문장에서 단어의 위치를 ​​나타내는 **위치 벡터**입니다.

모델은 이 추가 정보를 어떻게 가장 효과적으로 활용할지 스스로 판단할 것입니다.

위치 정보를 추가하는 가장 간단한 방법은 각 단어의 위치를 ​​임베딩 벡터에 연결하는 것입니다. 벡터에 "위치" 축을 추가하고, 순서상 첫 번째 단어는 0, 두 번째 단어는 1 등으로 값을 채우면 됩니다.

하지만 이 방법은 위치 값이 매우 큰 정수일 수 있기 때문에 이상적이지 않을 수 있습니다. 이는 임베딩 벡터의 값 범위를 왜곡할 수 있습니다. 아시다시피 신경망은 매우 큰 입력값이나 불연속적인 입력 분포를 좋아하지 않습니다.

"Attention is all you need"의 저자들은 단어 위치를 인코딩하기 위해 흥미로운 기법을 사용했습니다. **단어 임베딩에 위치에 따라 주기적으로 변하는 [-1, 1] 범위의 값을 가진 벡터를 추가**한 것입니다(이를 위해 **코사인 함수를 사용**했습니다). 이 기법은 작은 값으로 이루어진 벡터를 통해 넓은 범위의 모든 정수를 고유하게 특징화할 수 있는 방법을 제공합니다. 기발한 방법이지만, **더 간단하고 효과적인 방법**이 있습니다. **단어 인덱스를 임베딩하는 것과 같은 방식으로 위치 임베딩 벡터를 학습하는 것**입니다. 그런 다음 위치 임베딩을 해당 단어 임베딩에 추가하여 위치를 인식하는 단어 임베딩을 얻습니다. 이를 위치 임베딩이라고 합니다. 이제 구현해 보겠습니다.

In [40]:
from keras import ops

class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

우리는 이 PositionalEmbedding 레이어를 일반적인 Embedding 레이어처럼 사용할 것입니다. 이제 Transformer를 두 번째로 학습시키면서 실제로 어떻게 작동하는지 살펴보겠습니다.

In [41]:
hidden_dim = 256
intermediate_dim = 2056
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

위치 임베딩이 모델에 추가되었으니 다시 학습을 시도해 보겠습니다.

In [42]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=30, validation_data=val_ds)

Epoch 1/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 47s 31ms/step - accuracy: 0.3834 - loss: 1.4327 - val_accuracy: 0.5372 - val_loss: 0.9465
Epoch 2/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 31s 23ms/step - accuracy: 0.5695 - loss: 0.9001 - val_accuracy: 0.6157 - val_loss: 0.7479
Epoch 3/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.6416 - loss: 0.7008 - val_accuracy: 0.6476 - val_loss: 0.6699
Epoch 4/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 31s 24ms/step - accuracy: 0.6826 - loss: 0.5854 - val_accuracy: 0.6609 - val_loss: 0.6408
Epoch 5/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.7110 - loss: 0.5068 - val_accuracy: 0.6742 - val_loss: 0.6250
Epoch 6/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 31s 24ms/step - accuracy: 0.7334 - loss: 0.4494 - val_accuracy: 0.6807 - val_loss: 0.6165
Epoch 7/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 31s 24ms/step - accuracy: 0.7536 - loss: 0.4037 - val_accuracy: 0.6841 - val_loss: 0.6152
Epoch 8/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 31s 24ms/step - accuracy: 0.7697 -

**위치 정보를 모델에 다시 도입하니 결과가 훨씬 좋아**졌습니다. 다음 단어를 예측하는 데 **67%의 정확도**를 달성했습니다. 이는 **GRU 모델에 비해 눈에 띄게 향상된 수치**이며, 특히 **이 모델이 GRU 모델의 절반에 불과한 파라미터를 사용한다는 점**을 고려하면 더욱 인상적입니다.

이번 학습 과정에서 또 하나 중요한 점이 있습니다. **학습 속도가 RNN보다 훨씬 빠르다**는 것입니다. 각 **에포크에 걸리는 시간이 약 3분의 1로 단축**되었습니다. 파라미터 개수를 RNN 모델과 동일하게 하더라도 마찬가지일 것이며, 이는 **GRU 레이어의 반복적인 상태 전달을 제거한 덕분**입니다. **어텐션 메커니즘을 사용하면 학습 중에 반복적인 연산을 처리할 필요가 없으므로 GPU나 TPU에서 전체 어텐션 연산을 한 번에 처리**할 수 있습니다. 따라서 **Transformer는 가속기에서 더 빠른 학습 속도를 제공**합니다.

이제 새로 학습된 Transformer를 사용하여 생성 연산을 다시 실행해 보겠습니다. RNN 샘플링에 사용했던 것과 동일한 코드를 사용하면 됩니다.

In [43]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = transformer.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
I'm dying for a smoke.
[start] me muero por un cigarrillo [end]
-
Tom wants to go to the movies.
[start] tom quiere ir al cine [end]
-
I ate last night's leftovers for lunch.
[start] comí último escritorio la última noche [end]
-
I learned a lot about Australia.
[start] aprendí mucho de australia [end]
-
You've certainly been busy.
[start] en serio has estado ocupado [end]


생성 코드를 실행하면 다음과 같은 출력이 나타납니다.

주관적으로 볼 때, **트랜스포머는 GRU 기반 번역 모델보다 훨씬 뛰어난 성능**을 보여줍니다. 여전히 장난감 모델에 가깝지만, 훨씬 더 나은 장난감 모델입니다.

트랜스포머는 텍스트 처리 모델에 대한 관심이 폭발적으로 증가한 기반을 마련한 강력한 아키텍처입니다. 딥러닝 모델 중에서도 상당히 복잡한 구조를 가지고 있습니다. 이러한 구현 세부 사항들을 모두 살펴본 후에는, 모든 것이 다소 임의적인 것처럼 보일 수 있다는 반론이 나올 수도 있습니다. 너무나 많은 세부 사항들을 그대로 받아들여야 하는데, 이러한 레이어 선택과 구성이 최적이라는 것을 어떻게 확신할 수 있을까요?

답은 간단합니다. 최적이 아닙니다. 수년에 걸쳐 어텐션, 정규화, 위치 임베딩 등을 변경하여 트랜스포머 아키텍처를 개선하는 여러 가지 방안이 제시되었습니다. 오늘날 많은 새로운 모델들은 시퀀스 길이가 매우 길어짐에 따라 어텐션을 계산 복잡성이 낮은 다른 방식으로 대체하고 있습니다. 결국, 아마도 이 책을 읽을 때쯤에는 언어 모델링에 사용되는 주요 아키텍처로서 트랜스포머를 대체하는 무언가가 등장했을지도 모릅니다.

트랜스포머로부터 배울 수 있는 것들은 앞으로도 오랫동안 가치를 지닐 것입니다. 이 장의 마지막 부분에서는 트랜스포머가 왜 그렇게 효과적인지 논의할 것입니다. 하지만 머신러닝 분야 전체가 경험적 검증을 통해 발전해 왔다는 점을 기억할 필요가 있습니다. 어텐션 모델은 순환신경망(RNN)을 강화하려는 시도에서 탄생했고, 수많은 사람들이 수년간 시행착오를 거친 끝에 트랜스포머가 개발되었습니다. 이러한 과정이 아직 끝나지 않았다고 생각할 만한 이유는 거의 없습니다.

### Classification with a pretrained Transformer

"Attention is all you need"라는 글이 나온 후, 사람들은 Transformer 학습의 확장성이 얼마나 뛰어난지, 특히 기존 모델들과 비교했을 때 얼마나 큰 장점인지 주목하기 시작했습니다. 앞서 언급했듯이, **Transformer의 가장 큰 장점 중 하나는 RNN보다 학습 속도가 빠르다**는 점입니다. GPU나 TPU를 사용할 때 학습 과정에서 반복문이 필요 없다는 점은 매우 유리합니다.

또한 **Transformer는 데이터 요구량이 매우 높은 모델 아키텍처**입니다. 지난 섹션에서 이를 직접 경험해 보았습니다. RNN 번역 모델은 5 에포크 정도 학습 후 검증 성능이 정체된 반면, Transformer 모델은 30 에포크 학습 후에도 검증 점수가 계속 향상되었습니다.

이러한 관찰을 바탕으로 많은 사람들이 **더 많은 데이터, 레이어, 파라미터를 사용하여 Transformer를 확장하는 시도**를 했고, **그 결과는 매우 좋았**습니다. 이는 학습 비용이 수백만 달러에 달하지만 텍스트 도메인의 다양한 문제에서 훨씬 뛰어난 성능을 보이는 대규모 사전 학습 모델에 대한 수요 증가로 이어졌습니다.

텍스트 섹션의 마지막 코드 예제에서는 사전 학습된 Transformer 모델을 사용하여 IMDb 텍스트 분류 문제를 다시 살펴보겠습니다.

#### Pretraining a Transformer encoder

자연어 처리(NLP) 분야에서 인기를 얻은 최초의 사전 학습된 트랜스포머 중 하나는 **BERT(Bidirectional Encoder Representations from Transformers)**[2]였습니다. 이 논문과 모델은 "Attention Is All You Need"가 발표된 지 1년 후에 공개되었습니다. **모델 구조는 방금 구축한 번역 트랜스포머의 인코더 부분과 정확히 동일**합니다. 이 인코더 모델은 양방향성을 가지는데, 시퀀스의 모든 위치가 앞뒤 위치에 어텐션을 적용할 수 있다는 의미입니다. 즉, 입력 텍스트의 풍부한 표현을 계산하는 데는 적합하지만, 반복적인 생성 작업을 수행하도록 설계된 모델은 아닙니다.

**BERT는 1억에서 3억 개의 파라미터**를 가진 크기로 학습되었는데, 이는 방금 학습한 1400만 개의 파라미터를 가진 트랜스포머보다 훨씬 큽니다. 따라서 모델이 좋은 성능을 내려면 많은 양의 학습 데이터가 필요했습니다. 이를 위해 저자들은 **마스크 언어 모델링(Masked Language Modeling)이라는 고전적인 언어 모델링 방식을 변형하여 사용**했습니다. 모델을 사전 학습하기 위해 **텍스트 시퀀스에서 약 15%의 토큰을 특수한 [MASK] 토큰으로 대체**했습니다. 이 모델은 **학습 과정에서 원래 마스킹된 토큰 값을 예측하려고 시도**합니다. **고전적인 언어 모델(때때로 인과 언어 모델이라고도 함)이 p(토큰|이전 토큰)을 예측**하려고 하는 반면, **마스크 언어 모델은 p(토큰|주변 토큰)을 예측**하려고 합니다.

이 학습 방식은 **비지도 학습**입니다. 입력 텍스트에 대한 레이블이 필요하지 않습니다. 어떤 텍스트 시퀀스에서든 임의의 토큰을 선택하여 마스킹할 수 있습니다. 덕분에 저자들은 이 규모의 모델을 **학습하는 데 필요한 방대한 양의 텍스트 데이터를 쉽게 확보**할 수 있었습니다. 대부분의 데이터는 위키피디아에서 가져왔습니다.

BERT가 출시되었을 당시 사전 학습된 단어 임베딩을 사용하는 것은 이미 일반적인 관행이었습니다. 지난 장에서 이를 직접 살펴보았습니다. 하지만 전체 트랜스포머를 사전 학습함으로써 훨씬 강력한 기능을 구현할 수 있었습니다. 바로 주변 단어들의 맥락 속에서 단어 임베딩을 계산할 수 있게 된 것입니다. 트랜스포머 덕분에 당시에는 상상할 수 없었던 규모와 품질로 이러한 작업을 수행할 수 있었습니다.

BERT 개발자들은 방대한 양의 텍스트로 사전 학습된 이 모델을 활용하여 당시 여러 자연어 처리 벤치마크에서 최첨단 성능을 달성했습니다. 이는 대규모 사전 학습 모델을 사용하고 미세 조정은 최소한으로만 수행하는 방향으로 자연어 처리 분야에 뚜렷한 변화를 가져왔습니다. 이제 직접 시도해 보겠습니다.

#### Loading a pretrained Transformer

여기서는 BERT 대신 **Robustly Optimized BERT의 약자인 RoBERTa[3]라는 후속 모델**을 사용해 보겠습니다. RoBERTa는 BERT 아키텍처를 약간 단순화했지만, 가장 중요한 것은 성능 향상을 위해 더 많은 훈련 데이터를 사용했다는 점입니다. BERT는 주로 위키피디아에서 가져온 16GB의 영어 텍스트를 사용했습니다. **RoBERTa 개발자들은 웹 전체에서 160GB의 텍스트를 사용**했습니다. 당시 RoBERTa를 훈련하는 데 수십만 달러가 소요된 것으로 추정됩니다. 이러한 추가 훈련 데이터 덕분에 동일한 전체 매개변수 개수에서 모델 성능이 눈에 띄게 향상되었습니다.

사전 훈련된 모델을 사용하려면 몇 가지가 필요합니다.

* **일치하는 토크나이저** - 사전 훈련된 모델 자체와 함께 사용됩니다. 모든 텍스트는 사전 훈련 시와 동일한 방식으로 토큰화되어야 합니다. IMDb 리뷰의 단어가 사전 훈련 시와 다른 토큰 인덱스에 매핑되는 경우, 모델에서 각 토큰의 학습된 표현을 사용할 수 없습니다.
* **일치하는 모델 아키텍처** — 사전 학습된 모델을 사용하려면 사전 학습에 사용된 내부 연산을 정확하게 재현해야 합니다.
* **사전 학습된 가중치** — 이 가중치는 1,024개의 GPU와 수십억 개의 입력 단어를 사용하여 약 하루 동안 모델을 학습시켜 생성되었습니다.

토크나이저와 아키텍처 코드를 직접 재현하는 것은 그리 어렵지 않습니다. 모델 내부 구조는 이전에 구축했던 TransformerEncoder와 거의 정확히 일치합니다. 그러나 모델 구현을 일치시키는 것은 시간이 많이 소요되는 작업이므로, 이 책의 앞부분에서처럼 KerasHub 라이브러리를 사용하여 Keras용 사전 학습된 모델 구현에 접근할 수 있습니다.

KerasHub를 사용하여 RoBERTa 토크나이저와 모델을 로드해 보겠습니다. 특수 생성자 from_preset()을 사용하여 사전 학습된 모델의 가중치, 구성 및 토크나이저 자산을 디스크에서 로드할 수 있습니다. RoBERTa 논문과 함께 공개된 몇 가지 사전 학습된 체크포인트 중 가장 작은 RoBERTa 기본 모델을 로드하겠습니다.


In [44]:
import keras_hub

tokenizer = keras_hub.models.Tokenizer.from_preset("roberta_base_en")
backbone = keras_hub.models.Backbone.from_preset("roberta_base_en")

100%|██████████| 445/445 [00:00<00:00, 846kB/s]


100%|██████████| 686/686 [00:00<00:00, 776kB/s]


100%|██████████| 0.99M/0.99M [00:00<00:00, 2.80MB/s]


100%|██████████| 446k/446k [00:00<00:00, 1.60MB/s]


100%|██████████| 474M/474M [00:13<00:00, 36.1MB/s]


토크나이저는 예상대로 텍스트를 정수 시퀀스로 변환합니다. 지난 장에서 만들었던 SubWordTokenizer를 기억하시나요? RoBERTa의 토크나이저는 모든 언어의 유니코드 문자를 처리할 수 있도록 약간 수정된 것을 제외하면 거의 동일합니다.

RoBERTa의 사전 학습 데이터셋 규모를 고려할 때, 서브워드 토큰화는 필수적입니다. 문자 수준 토크나이저를 사용하면 입력 시퀀스가 ​​너무 길어져 모델 학습 비용이 크게 증가합니다. 단어 수준 토크나이저를 사용하려면 웹에서 사용되는 수백만 개의 문서에 있는 모든 단어를 포괄하기 위해 방대한 어휘집이 필요합니다. 단어를 충분히 포괄하려면 어휘집 크기가 기하급수적으로 커져 트랜스포머 맨 앞의 임베딩 레이어가 감당할 수 없을 정도로 커집니다. 서브워드 토크나이저를 사용하면 모델은 단 5만 개의 단어로 구성된 어휘집으로 모든 단어를 처리할 수 있습니다.

In [45]:
tokenizer("The quick brown fox")

Array([  133,  2119,  6219, 23602], dtype=int32)

방금 불러온 백본(Backbone)은 무엇일까요? 8장에서 살펴본 것처럼, 백본은 컴퓨터 비전에서 입력 이미지를 잠재 공간으로 매핑하는 네트워크를 가리키는 용어입니다. 기본적으로 예측 기능을 제외한 비전 모델이라고 할 수 있죠. KerasHub에서는 백본을 특정 작업에 특화되지 않은 사전 학습된 모델로 정의합니다. 방금 불러온 모델은 입력 시퀀스를 받아 (batch_size, sequence_length, 768) 형태의 출력 시퀀스로 임베딩하지만, 특정 손실 함수에 맞춰 설정되어 있지는 않습니다. 따라서 문장 분류, 특정 정보를 포함하는 텍스트 구간 식별, 품사 식별 등 다양한 하위 작업에 활용할 수 있습니다.

다음으로, 이 백본에 분류 기능을 추가하는 헤드를 연결하여 IMDb 리뷰 분류 미세 조정에 특화시키겠습니다. 마치 드라이버에 여러 종류의 헤드를 연결하는 것과 같습니다. 한 작업에는 십자 드라이버를, 다른 작업에는 일자 드라이버를 연결하는 것처럼 말이죠.

이제 백본을 자세히 살펴보겠습니다. 여기서는 RoBERTa의 가장 작은 버전을 사용했지만, 여전히 1억 2400만 개의 매개변수를 가지고 있으며, 이는 이 책에서 사용한 모델 중 가장 큰 규모입니다.

In [46]:
backbone.summary(line_length=80)

Model: "roberta_backbone"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ token_ids             │ (None, None)      │           0 │ -                  │
│ (InputLayer)          │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embeddings            │ (None, None, 768) │  38,996,736 │ token_ids[0][0]    │
│ (TokenAndPositionEmb… │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embeddings_layer_norm │ (None, None, 768) │       1,536 │ embeddings[0][0]   │
│ (LayerNormalization)  │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embeddings_dropout    │ (None, None, 768) │           0 │ embeddings_layer_… │
│ (Dropout)             │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ padding_mask          │ (None, None)      │           0 │ -                  │
│ (InputLayer)          │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_0   │ (None, None, 768) │   7,087,872 │ embeddings_dropou… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_1   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_2   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_3   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_4   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_5   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_6   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_7   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_8   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_9   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_10  │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │      

 Total params: 124,052,736 (473.22 MB)

 Trainable params: 124,052,736 (473.22 MB)

 Non-trainable params: 0 (0.00 B)

RoBERTa는 12개의 Transformer 인코더 레이어를 서로 쌓아서 사용합니다. 이는 저희 번역 모델보다 훨씬 발전된 것입니다!

#### Preprocessing IMDb movie reviews

14장에서 사용했던 IMDb 로딩 코드를 변경 없이 그대로 재사용할 수 있습니다. 이 코드는 영화 리뷰 데이터를 train_dir과 test_dir에 다운로드하고, 검증 데이터셋을 val_dir로 분할합니다.

In [47]:
import os, pathlib, shutil, random

zip_path = keras.utils.get_file(
    origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
    fname="imdb",
    extract=True,
)

imdb_extract_dir = pathlib.Path(zip_path) / "aclImdb"
train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

shutil.copytree(imdb_extract_dir / "test", test_dir, dirs_exist_ok=True)

val_percentage = 0.2
for category in ("neg", "pos"):
    src_dir = imdb_extract_dir / "train" / category
    src_files = os.listdir(src_dir)
    random.Random(1337).shuffle(src_files)
    num_val_samples = int(len(src_files) * val_percentage)

    os.makedirs(train_dir / category, exist_ok=True)
    os.makedirs(val_dir / category, exist_ok=True)
    for index, file in enumerate(src_files):
        if index < num_val_samples:
            shutil.copy(src_dir / file, val_dir / category / file)
        else:
            shutil.copy(src_dir / file, train_dir / category / file)

84125825/84125825 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [48]:
from keras.utils import text_dataset_from_directory

batch_size = 16
train_ds = text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds = text_dataset_from_directory(val_dir, batch_size=batch_size)
test_ds = text_dataset_from_directory(test_dir, batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


데이터를 불러온 후, 우리는 다시 20,000개의 영화 리뷰로 구성된 학습 데이터셋과 5,000개의 영화 리뷰로 구성된 검증 데이터셋을 갖게 되었습니다.

분류 모델을 미세 조정하기 전에, 불러온 RoBERTa 토크나이저를 사용하여 영화 리뷰 데이터를 토큰화해야 합니다. 사전 학습 과정에서 RoBERTa는 번역 모델에서 했던 것과 유사하게 토큰들을 특정 형태의 "패킹" 방식으로 시퀀스로 묶었습니다. 각 시퀀스는 <s> 토큰으로 시작하고 </s> 토큰으로 끝나며, 그 뒤에는 다음과 같이 여러 개의 <pad> 토큰이 붙습니다.
```
[
    ["<s>", "the", "quick", "brown", "fox", "jumped", ".", "</s>"],
    ["<s>", "the", "panda", "slept", ".", "</s>", "<pad>", "<pad>"],
]
```
사전 학습에 사용된 토큰 순서를 최대한 일치시키는 것이 중요합니다. 이렇게 하면 모델이 더 빠르고 정확하게 학습됩니다. KerasHub는 이러한 토큰 패킹을 위한 StartEndPacker라는 레이어를 제공합니다. 이 레이어는 시작, 끝 및 패딩 토큰을 추가하고, 필요한 경우 긴 시퀀스를 지정된 시퀀스 길이로 잘라냅니다. 이제 이 레이어를 토크나이저와 함께 사용해 보겠습니다.

In [49]:
def preprocess(text, label):
    packer = keras_hub.layers.StartEndPacker(
        sequence_length=512,
        start_value=tokenizer.start_token_id,
        end_value=tokenizer.end_token_id,
        pad_value=tokenizer.pad_token_id,
        return_padding_mask=True,
    )
    token_ids, padding_mask = packer(tokenizer(text))
    return {"token_ids": token_ids, "padding_mask": padding_mask}, label

preprocessed_train_ds = train_ds.map(preprocess)
preprocessed_val_ds = val_ds.map(preprocess)
preprocessed_test_ds = test_ds.map(preprocess)

전처리된 단일 배치를 살펴보겠습니다.

In [50]:
next(iter(preprocessed_train_ds))

({'token_ids': <tf.Tensor: shape=(16, 512), dtype=int32, numpy=
  array([[    0,   100,  3996, ...,     1,     1,     1],
         [    0,  3721, 13766, ...,   963,    36,     2],
         [    0,   100,  1166, ...,     1,     1,     1],
         ...,
         [    0,   113,   133, ...,     1,     1,     1],
         [    0,   100,   218, ...,     1,     1,     1],
         [    0,  6179,   222, ...,     1,     1,     1]], dtype=int32)>,
  'padding_mask': <tf.Tensor: shape=(16, 512), dtype=bool, numpy=
  array([[ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ...,  True,  True,  True],
         [ True,  True,  True, ..., False, False, False],
         ...,
         [ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False],
         [ True,  True,  True, ..., False, False, False]])>},
 <tf.Tensor: shape=(16,), dtype=int32, numpy=array([1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0], dtype=int32)>)

입력 데이터 전처리가 완료되었으므로 이제 모델을 미세 조정할 준비가 되었습니다.

#### Where does pretraining data come from?

트랜스포머는 데이터를 많이 필요로 하는 모델입니다. 딥러닝 역사상 전례 없는 규모의 입력 데이터를 제공할수록 성능이 향상됩니다. 최초의 트랜스포머는 수백만 개의 토큰으로 구성된 번역 데이터셋으로 학습되었습니다. 이후 수십억 개, 그리고 현재는 수조 개의 토큰으로 학습되고 있습니다. 엄청난 양의 단어들이죠.

그렇다면 이 모든 데이터는 어디에서 오는 걸까요? 답은 시간이 지남에 따라 바뀌었지만, 간단히 말하면 인터넷입니다. 또 다른 답은 이 부분이 점점 비밀이 되어가고 있다는 것입니다. 기업들은 모델 학습에 사용한 정확한 데이터셋이나 데이터 소스의 조합을 공개하지 않는 경우가 많습니다.

시대별로 사용된 몇 가지 사전 학습 데이터셋을 살펴보겠습니다.

* **최초의 트랜스포머는 400만 쌍의 문장으로 구성된 유명한 영어-독일어 번역 데이터셋으로 학습**되었습니다.
* **BERT는 영어 위키백과 데이터셋과 7,000권의 자가 출판 서적 데이터셋**을 사용했습니다.
* **ChatGPT의 전신인 GPT2는 Reddit에서 나가는 링크를 따라가며 데이터셋을 수집**했습니다.
* **Meta에서 출시한 사전 학습된 Transformer 모델인 Llama의 최신 버전은 "공개적으로 이용 가능한 소스에서 가져온 15조 개의 토큰 데이터"로 학습**되었습니다. 데이터의 정확한 구성을 모호하게 남겨두는 것이 점점 더 일반화되고 있습니다.

다음 장에서는 사전 학습 데이터 소스의 정확한 조합이 얼마나 중요한지 살펴보겠습니다. 가능하면 모델의 데이터 출처에 항상 주의를 기울이는 것이 좋습니다. 데이터의 출처는 모델의 편향과 성능에 영향을 미치기 때문입니다.

#### Fine-tuning a pretrained Transformer

**영화 리뷰 예측을 위해 백본을 미세 조정**하기 전에, 먼저 **이진 분류 레이블을 출력하도록 업데이트**해야 합니다. **백본은 (배치 크기, 시퀀스 길이, 768)** 형태의 전체 시퀀스를 출력하는데, 각 768차원 벡터는 주변 단어와의 문맥 속에서 입력 단어를 나타냅니다. 레이블을 예측하기 전에 이 시퀀스를 샘플당 하나의 벡터로 압축해야 합니다.

한 가지 방법은 전체 시퀀스에 걸쳐 평균 풀링이나 최대 풀링을 수행하여 모든 토큰 벡터의 평균을 계산하는 것입니다. 하지만 **첫 번째 토큰의 표현을 풀링 값으로 사용하는 것이 조금 더 효과적**입니다. 이는 모델의 어텐션 메커니즘 때문입니다. **최종 인코더 레이어의 첫 번째 위치는 시퀀스의 다른 모든 위치에 어텐션을 적용하여 정보를 가져올 수 있습**니다. 따라서 평균을 내는 것과 같은 단순한 풀링 방식 대신, 어텐션을 통해 시퀀스 전체에 걸쳐 문맥적으로 정보를 풀링할 수 있습니다.

이제 백본에 **분류 헤드를 추가**해 보겠습니다. 또한 **출력 예측을 생성하기 전에 비선형성을 포함하는 최종 Dense 투영을 하나 더 추가**할 것입니다.

In [51]:
inputs = backbone.input
x = backbone(inputs)
x = x[:, 0, :]
x = layers.Dropout(0.1)(x)
x = layers.Dense(768, activation="relu")(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
classifier = keras.Model(inputs, outputs)

이제 IMDb 데이터셋을 사용하여 모델을 미세 조정하고 평가할 준비가 되었습니다.

In [52]:
classifier.compile(
    optimizer=keras.optimizers.Adam(5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
classifier.fit(
    preprocessed_train_ds,
    validation_data=preprocessed_val_ds,
)

1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2361s 2s/step - accuracy: 0.9100 - loss: 0.2368 - val_accuracy: 0.9178 - val_loss: 0.1961


마지막으로 학습된 모델을 평가해 보겠습니다.

In [53]:
classifier.evaluate(preprocessed_test_ds)

1563/1563 ━━━━━━━━━━━━━━━━━━━━ 937s 596ms/step - accuracy: 0.9284 - loss: 0.1819


[0.1819167286157608, 0.9283599853515625]

단 한 번의 학습 에포크 만에 우리 모델은 93%의 정확도를 달성했는데, 이는 지난 장에서 기록했던 90%에서 눈에 띄게 향상된 수치입니다. 물론, 이 모델은 이전에 구축했던 간단한 바이그램 분류기보다 훨씬 더 많은 비용이 들지만, 이렇게 큰 모델을 사용하는 데에는 분명한 이점이 있습니다. 게다가 이 모든 성과는 비교적 작은 규모의 RoBERTa 모델로 달성한 것입니다. 3억 개의 파라미터를 가진 더 큰 모델을 사용한다면 95% 이상의 정확도를 얻을 수 있을 것입니다.

### What makes the Transformer effective?

2013년 구글에서 토마스 미콜로프와 그의 동료들은 놀라운 사실을 발견했습니다. 그들은 지난 장에서 다룬 CBOW(Continuous Bag of Words) 임베딩과 유사한 "Word2Vec"이라는 사전 학습된 임베딩을 구축하고 있었습니다. CBOW 모델과 마찬가지로, 그들의 학습 목표는 단어 간의 상관관계를 임베딩 공간에서의 거리 관계로 변환하는 것이었습니다. 즉, 어휘의 각 단어에 벡터를 연결하고, 자주 함께 나타나는 단어를 나타내는 벡터 간의 내적(코사인 근접도)이 1에 가깝고, 드물게 함께 나타나는 단어를 나타내는 벡터 간의 내적이 0에 가깝도록 최적화했습니다.

그들은 결과적으로 얻어진 임베딩 공간이 의미적 유사성뿐만 아니라 훨씬 더 많은 것을 포착한다는 것을 발견했습니다. 그것은 일종의 "단어 연산"과 같은, 새로운 형태의 학습을 특징으로 했습니다. 공간에는 여러 남성 명사에 더해 여성 명사에 가장 가까운 값을 얻을 수 있는 벡터가 존재했습니다. 예를 들어 V(king) - V(man) + V(woman) = V(queen)과 같은 성별 벡터였습니다. 이는 매우 놀라운 발견이었습니다. 모델이 명시적으로 이러한 기능을 학습하도록 설계된 적이 없었기 때문입니다. 복수형 벡터, 야생 동물 이름에서 가장 유사한 애완동물 이름을 찾는 벡터 등 이러한 마법의 벡터가 수십 개나 있는 듯했습니다.

약 10년 후, 우리는 대규모 사전 학습된 Transformer 모델의 시대에 살고 있습니다. 겉으로 보기에는 이러한 모델들이 초기 Word2Vec 모델과는 완전히 다른 것처럼 보입니다. Transformer는 완벽하게 유창한 언어를 생성할 수 있는데, 이는 Word2Vec이 전혀 불가능했던 일입니다. 다음 장에서 살펴보겠지만, 이러한 모델은 거의 모든 주제에 대해 해박한 지식을 가진 것처럼 보일 수 있습니다. 하지만 사실 Transformer는 Word2Vec과 많은 공통점을 가지고 있습니다.

두 모델 모두 토큰(단어 또는 하위 단어)을 벡터 공간에 임베딩하려고 합니다. 두 모델 모두 임베딩 공간을 학습하는 데 있어 동일한 기본 원리를 사용합니다. 즉, 함께 나타나는 토큰은 임베딩 공간에서 가까운 위치에 있게 됩니다. 토큰을 비교하는 데 사용되는 거리 함수는 두 경우 모두 코사인 거리입니다. 임베딩 공간의 차원조차도 유사합니다. 각 단어를 표현하기 위해 1,000에서 10,000 사이의 차원을 가진 벡터를 사용합니다.

여기서 다음과 같은 의문이 들 수 있습니다. 트랜스포머는 시퀀스에서 누락된 단어를 예측하도록 훈련된 것이지, 임베딩 공간에서 토큰을 그룹화하도록 훈련된 것이 아닙니다. 언어 모델의 손실 함수가 Word2Vec의 목표인 함께 나타나는 토큰 간의 내적 최대화와 어떤 관련이 있을까요? 그 답은 어텐션 메커니즘에 있습니다.

어텐션은 트랜스포머 아키텍처에서 가장 중요한 구성 요소입니다. 어텐션은 이전 임베딩 공간에서 토큰 임베딩을 선형적으로 재조합하여 새로운 토큰 임베딩 공간을 학습하는 메커니즘입니다. 이때, 이미 서로 "가까운" 토큰(즉, 내적이 더 큰 토큰)에 더 큰 가중치를 부여합니다. 어텐션은 이미 가까운 토큰들의 벡터를 서로 끌어당기는 경향이 있으며, 시간이 지남에 따라 토큰 상관 관계가 임베딩 근접 관계(코사인 거리 기준)로 변환되는 공간을 생성합니다. 트랜스포머는 이전 임베딩 공간의 요소들을 재조합하여 점진적으로 정제된 일련의 임베딩 공간을 학습하는 방식으로 작동합니다.

어텐션은 트랜스포머에 두 가지 중요한 속성을 제공합니다.

* 트랜스포머가 학습하는 임베딩 공간은 의미적으로 연속적입니다. 즉, 임베딩 공간에서 비트만큼 이동해도 해당 토큰의 사람이 인식하는 의미는 비트만큼만 변화합니다. Word2Vec 공간 또한 이러한 속성을 보였습니다.
* 이들이 학습하는 임베딩 공간은 의미론적으로 보간적입니다. 즉, 임베딩 공간에서 두 점 사이의 중간점을 취하면 해당 토큰 사이의 "중간 의미"를 나타내는 점이 생성됩니다. 이는 각 새로운 임베딩 공간이 이전 공간의 벡터 사이를 보간하여 구축된다는 사실에서 비롯됩니다.

이는 뇌의 학습 방식과 완전히 다르지는 않습니다. 뇌의 핵심 학습 원리는 헤비안 학습, 즉 "함께 발화하는 뉴런은 함께 연결된다"는 것입니다. 신경 발화 사건(행동이나 지각 입력을 나타낼 수 있음) 간의 상관 관계는 뇌 네트워크에서 근접 관계로 변환되는데, Transformer와 Word2Vec이 상관 관계를 벡터 근접 관계로 변환하는 것과 유사합니다. 둘 다 정보 공간의 지도입니다.

물론 Word2Vec과 Transformer 사이에는 상당한 차이점이 있습니다. Word2Vec은 텍스트 생성 샘플링을 위해 설계된 것이 아닙니다. Transformer는 훨씬 더 큰 규모로 확장될 수 있으며 훨씬 더 복잡한 변환을 인코딩할 수 있습니다. 문제는 Word2Vec이 오늘날의 언어 모델과 비교했을 때, MNIST 픽셀에 대한 로지스틱 회귀 모델이 최첨단 컴퓨터 비전 모델과 비교되는 것과 같은 수준의, 말 그대로 장난감 모델이라는 점입니다. 기본적인 원리는 대부분 동일하지만, 장난감 모델은 의미 있는 표현력을 결여하고 있습니다. Word2Vec은 심지어 심층 신경망조차 아니었습니다. 얕은 단일 레이어 구조를 가지고 있었습니다. 반면, 오늘날의 Transformer 모델은 지금까지 훈련된 어떤 모델보다도 가장 강력한 표현력을 자랑합니다. 수십 개의 어텐션 및 피드포워드 레이어가 쌓여 있고, 파라미터 수는 수십억 개에 달합니다.

Word2Vec과 마찬가지로 Transformer는 토큰을 벡터 공간으로 구성하는 과정에서 유용한 의미 기능을 자연스럽게 학습합니다. 하지만 향상된 표현력과 훨씬 더 정교해진 자기회귀 최적화 목표 덕분에, 우리는 더 이상 성별 벡터나 복수형 벡터와 같은 선형 변환에만 국한되지 않습니다. 트랜스포머는 임의로 복잡한 벡터 함수를 저장할 수 있습니다. 사실, 너무 복잡해서 함수라기보다는 벡터 프로그램이라고 부르는 것이 더 정확할 정도입니다.

Word2Vec을 사용하면 복수형(고양이) → 고양이들(cats)이나 남성형(king) → 여성형(queen)과 같은 기본적인 변환을 할 수 있었습니다. 반면, 대규모 트랜스포머 모델은 셰익스피어 스타일로 시를 쓰는 마법 같은 작업, 예를 들어 새로운 시를 쓰는 마법 같은 작업도 수행할 수 있습니다. 하나의 모델에 수백만 개의 이러한 프로그램을 저장할 수도 있습니다.

트랜스포머는 데이터베이스와 유사하다고 볼 수 있습니다. 트랜스포머는 사용자가 전달하는 토큰을 통해 검색할 수 있는 정보를 저장합니다. 하지만 트랜스포머와 데이터베이스 사이에는 두 가지 중요한 차이점이 있습니다.

첫 번째 차이점은 트랜스포머가 연속적이고 보간적인 데이터베이스라는 점입니다. 데이터는 개별 항목들의 집합으로 저장되는 대신, 벡터 공간, 즉 곡선으로 저장됩니다. 이 곡선 위를 이동하며(앞서 논의했듯이 의미론적으로 연속적입니다) 인접한 관련 지점들을 탐색할 수 있습니다. 또한, 곡선 상에서 서로 다른 데이터 지점 사이를 보간하여 중간값을 찾을 수 있습니다. 이는 데이터베이스에 입력한 데이터보다 훨씬 더 많은 데이터를 검색할 수 있다는 것을 의미합니다. 물론 모든 데이터가 정확하거나 의미 있는 것은 아닙니다. 보간은 일반화를 가능하게 하지만, 잘못된 해석을 초래할 수도 있는데, 이는 오늘날 학습되는 생성형 언어 모델이 직면한 중요한 문제입니다.

두 번째 차이점은 트랜스포머가 단순히 데이터만 저장하는 것이 아니라는 점입니다. 인터넷에서 수집한 수십만 개의 문서를 기반으로 학습된 RoBERTa와 같은 모델에는 사실, 장소, 인물, 날짜, 사물, 관계 등 방대한 데이터가 저장되어 있습니다. 하지만 트랜스포머는 프로그램 데이터베이스이기도 하며, 어쩌면 프로그램의 주요 데이터베이스라고 할 수 있습니다.

이 프로그램들은 여러분이 익숙하게 다루던 프로그램과는 다릅니다. 파이썬 프로그램처럼 데이터를 단계적으로 처리하는 일련의 기호 명령으로 이루어진 것이 아닙니다. 오히려 이러한 벡터 프로그램은 잠재 임베딩 공간을 자기 자신으로 매핑하는 고도로 비선형적인 함수입니다. Word2Vec의 마법 벡터와 유사하지만 훨씬 더 복잡합니다.

다음 장에서는 트랜스포머 모델을 완전히 새로운 규모로 확장할 것입니다. 모델은 수십억 개의 매개변수를 사용하고 수조 개의 단어로 학습될 것입니다. 이러한 모델의 출력은 마치 마법처럼 느껴질 수 있습니다. 마치 모델 안에 지능적인 연산자가 앉아서 모든 것을 조종하는 것처럼 말입니다. 하지만 이러한 모델은 근본적으로 보간적이라는 점을 기억하는 것이 중요합니다. 어텐션 메커니즘 덕분에 모델은 영어로 작성된 모든 텍스트의 상당 부분을 차지하는 보간 임베딩 공간을 학습합니다. 이 임베딩 공간을 탐색하면 흥미롭고 예상치 못한 일반화를 얻을 수 있지만, 인간 수준의 지능에 근접하는 근본적으로 새로운 것을 합성할 수는 없습니다.

### Summary

* 언어 모델은 특정 확률 분포(p(토큰|이전 토큰))를 학습하는 모델입니다.
  - 언어 모델은 광범위하게 응용되지만, 가장 중요한 점은 루프에서 모델을 호출하여 텍스트를 생성할 수 있다는 것입니다. 즉, 한 시점의 출력 토큰이 다음 시점의 입력 토큰이 됩니다.
  - 마스크드 언어 모델은 관련 확률 분포(p(토큰|주변 토큰))를 학습하며 텍스트 및 개별 토큰 분류에 유용할 수 있습니다.
  - 시퀀스-투-시퀀스 언어 모델은 대상 시퀀스의 이전 토큰과 완전히 별개의 고정된 소스 시퀀스가 ​​모두 주어졌을 때 다음 토큰을 예측하도록 학습합니다. 시퀀스-투-시퀀스 모델은 번역 및 질의응답과 같은 문제에 유용합니다.
  - 시퀀스-투-시퀀스 모델은 일반적으로 두 개의 구성 요소로 이루어져 있습니다. 인코더는 소스 시퀀스의 표현을 계산하고, 디코더는 이 표현을 입력으로 받아 이전 토큰을 기반으로 대상 시퀀스의 다음 토큰을 예측합니다.

* 어텐션은 모델이 현재 처리 중인 토큰의 컨텍스트에 따라 시퀀스 내 어디에서든 선택적으로 정보를 가져올 수 있도록 하는 메커니즘입니다.
  - 어텐션은 텍스트에서 장거리 의존성으로 인해 RNN이 겪는 문제를 해결합니다.
  - 어텐션은 두 벡터의 내적을 계산하여 어텐션 점수를 산출하는 방식으로 작동합니다. 임베딩 공간에서 서로 가까운 벡터는 어텐션 메커니즘에서 함께 합산됩니다.
* 트랜스포머는 시퀀스 전체에 정보를 전달하는 유일한 메커니즘으로 어텐션을 사용하는 시퀀스 모델링 아키텍처입니다.
  - 트랜스포머는 어텐션과 2계층 피드포워드 네트워크를 번갈아 쌓아 올려 작동합니다.
  - 트랜스포머는 많은 매개변수와 방대한 훈련 데이터를 처리하면서도 언어 모델링 문제에서 정확도를 향상시킬 수 있습니다.
  - RNN과 달리 트랜스포머는 훈련 시 시퀀스 길이만큼의 반복문이 없어 여러 대의 컴퓨터에서 병렬로 모델을 훈련하기가 훨씬 쉽습니다.
  - 트랜스포머 인코더는 양방향 어텐션을 사용하여 시퀀스의 풍부한 표현을 구축합니다.
  - 트랜스포머 디코더는 인과적 어텐션을 사용하여 언어 모델에서 다음 단어를 예측합니다.